# Измерение стоимости полного и оконного сопоставления

Самодостаточный ноутбук: вспомогательные функции находятся в ячейках, локальные `.py` не импортируются. Нужны NumPy и отдельно собранный модуль Betti-Matching-3D; инструкция — `docs/BUILD.md`. По умолчанию `MODE='quick'`: маленький синтетический вход, два повтора, проверка конвейера.

Для `MODE='full'` нужен точный архив вероятностных карт и разметки; см. `docs/INPUT_DATA.md`. Архив доступен в [публичном релизе](https://github.com/sunightflower/Windowed-Betti-Matching-Cost/releases/tag/predictions). Скачайте `H1_preds.npz` под именем `predictions.npz` в рабочий каталог ноутбука; команды и SHA-256 приведены в [INPUT_DATA.md](../docs/INPUT_DATA.md). Сохранённые таблицы можно воспроизвести без него через первый ноутбук.

In [1]:
from pathlib import Path
import sys, json, time, hashlib, itertools, platform
from datetime import datetime, timezone
import numpy as np
import psutil

# quick: только маленький синтетический вход; full: заданный архив предсказаний.
MODE = 'quick'
STAGE = 'test'  # 'calibration' или 'test' в полном режиме
INPUT_PATH = Path('predictions.npz')
BM_MODULE_DIR = ''  # при необходимости: каталог собранного betti_matching, без имени файла
SAVE_RESULTS = False
OUTPUT = Path('new_measurements')
VERIFY_REFERENCE_INPUT = True
RUN_PARALLEL = True
RUN_DISCREPANCY = False  # дополнительная функция и градиент: вне таймеров
PARALLEL_CHUNK = 12
if BM_MODULE_DIR:
    sys.path.insert(0, str(Path(BM_MODULE_DIR).expanduser().resolve()))
import betti_matching as bm

def sha256_file(path):
    """Контрольная сумма файла без вывода локального пути."""
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1<<20),b''):h.update(block)
    return h.hexdigest()

def machine_environment():
    """Сведения о текущей машине и фактически загруженном модуле."""
    model=platform.processor().strip() or None
    if sys.platform=='win32':
        try:
            import winreg
            with winreg.OpenKey(winreg.HKEY_LOCAL_MACHINE,
                                r'HARDWARE\DESCRIPTION\System\CentralProcessor\0') as key:
                model=winreg.QueryValueEx(key,'ProcessorNameString')[0].strip()
        except OSError:
            pass
    elif sys.platform.startswith('linux'):
        try:
            for line in Path('/proc/cpuinfo').read_text().splitlines():
                if line.startswith('model name'):
                    model=line.split(':',1)[1].strip()
                    break
        except OSError:
            pass
    return {'python':platform.python_version(),'os':platform.platform(),
            'numpy':np.__version__,'cpu':model,
            'cores_physical':psutil.cpu_count(logical=False),
            'cores_logical':psutil.cpu_count(logical=True),
            'ram_gb':round(psutil.virtual_memory().total/(1024**3),3),
            'bm_binary_sha256':sha256_file(bm.__file__)}

def crop(array,box):
    """Извлечь прямоугольную область по полуоткрытым границам."""
    return array[tuple(slice(box[i],box[i+1]) for i in [0,2,4])]

def voxel_count(box):
    """Число вокселей области."""
    return int(np.prod([box[i+1]-box[i] for i in [0,2,4]]))

def windows(shape,side,margin):
    """Полное покрытие: собственные части без пересечений, поля обрезаны границами."""
    result=[]
    for start in itertools.product(*[range(0,n,side) for n in shape]):
        core=tuple(x for p,n in zip(start,shape) for x in [p,min(n,p+side)])
        box=tuple(x for p,n in zip(start,shape) for x in [max(0,p-margin),min(n,p+side+margin)])
        result.append({'core':core,'box':box})
    return result

def call_matching(p,y,box):
    """Извлечение, преобразование float64 и сопоставление внутри таймера."""
    f=np.ascontiguousarray(1.0-crop(p,box).astype(np.float64))
    g=np.ascontiguousarray(1.0-crop(y,box).astype(np.float64))
    return bm.compute_matching(f,g,include_input1_unmatched_pairs=True,
                               include_input2_unmatched_pairs=False)

def timed_call(p,y,box):
    """Измерить wall и CPU; результат остаётся жив до остановки таймера."""
    c0=time.process_time();t0=time.perf_counter()
    result=call_matching(p,y,box)
    return time.perf_counter()-t0,time.process_time()-c0,result

def warmup(p,y,boxes):
    """До трёх представителей 40 наиболее частых форм и область не больше 64³."""
    groups={}
    for b in boxes:
        groups.setdefault(tuple(b[i+1]-b[i] for i in [0,2,4]),[]).append(b)
    for _,group in sorted(groups.items(),key=lambda item:-len(item[1]))[:40]:
        for box in group[:1 if voxel_count(group[0])>10**6 else 3]:call_matching(p,y,box)
    call_matching(p,y,tuple(x for n in p.shape for x in [0,min(64,n)]))

def calibration_passes(p,y,boxes,repeats):
    """Сохранить каждый замер по окну и медианы; порядок seed=1000+r."""
    warmup(p,y,boxes)
    wall=np.zeros((len(boxes),repeats));cpu=wall.copy()
    for r in range(repeats):
        for i in np.random.default_rng(1000+r).permutation(len(boxes)):
            dt,dc,result=timed_call(p,y,boxes[i]);wall[i,r]=dt;cpu[i,r]=dc
            del result
    return [{'box':list(b),'voxels':voxel_count(b),'wall_passes_s':wall[i].tolist(),
             'cpu_passes_s':cpu[i].tolist(),'wall_median_s':float(np.median(wall[i])),
             'cpu_median_s':float(np.median(cpu[i]))} for i,b in enumerate(boxes)]

def direct_passes(p,y,variants,repeats):
    """Прямые проходы с циклическим сдвигом вариантов и медианой всего прохода."""
    boxes=list(dict.fromkeys(reg['box'] for regs in variants.values() for reg in regs))
    warmup(p,y,boxes)
    names=list(variants);known=set();tot={name:[] for name in names};cpus={name:[] for name in names}
    # В первом проходе сохраняем результаты до остановки таймера, как в исходной процедуре.
    for r in range(repeats):
        order=names[r%len(names):]+names[:r%len(names)]
        for name in order:
            pending=[];c0=time.process_time();t0=time.perf_counter()
            for reg in variants[name]:
                dt,dc,result=timed_call(p,y,reg['box'])
                if reg['box'] not in known:pending.append((reg['box'],result))
                else:del result
            tot[name].append(time.perf_counter()-t0);cpus[name].append(time.process_time()-c0)
            for box,result in pending:known.add(box)
            pending.clear()
    return {name:{'wall_passes_s':tot[name],'cpu_passes_s':cpus[name],
                  'wall_median_s':float(np.median(tot[name])),
                  'cpu_median_s':float(np.median(cpus[name]))} for name in names}

def parallel_passes(p,y,regs,repeats,chunk):
    """Пакеты окон через интерфейс модуля; chunk не ограничивает все внутренние потоки."""
    def run():
        for i in range(0,len(regs),chunk):
            batch=regs[i:i+chunk]
            fs=[np.ascontiguousarray(1.0-crop(p,r['box']).astype(np.float64)) for r in batch]
            gs=[np.ascontiguousarray(1.0-crop(y,r['box']).astype(np.float64)) for r in batch]
            bm.compute_matching(fs,gs,include_input1_unmatched_pairs=True,include_input2_unmatched_pairs=False)
    run();values=[]
    for _ in range(repeats):
        t0=time.perf_counter();run();values.append(time.perf_counter()-t0)
    return {'wall_passes_s':values,'wall_median_s':float(np.median(values))}

## Входы и проверка геометрии

In [2]:
assert MODE in ['quick','full'] and STAGE in ['calibration','test']
input_hash=None
if MODE=='quick':
    shape=(24,32,24);z,y,x=np.indices(shape)
    target=((z-12)**2+(y-16)**2+(x-12)**2)<7**2
    rng=np.random.default_rng(123)
    pred=np.clip(.1+.8*target+rng.normal(0,.08,shape),0,1).astype(np.float16).astype(np.float32)
    inputs={'synthetic':pred};repeats=2;geometries=[(8,2),(16,2)]
else:
    if not INPUT_PATH.is_file():
        raise FileNotFoundError('Нужен архив predictions.npz; формат и статус данных описаны в docs/INPUT_DATA.md')
    input_hash=sha256_file(INPUT_PATH)
    if VERIFY_REFERENCE_INPUT:
        assert input_hash=='0a200863ff8b01757132ae390443c51a0406dd3fcc6ce7e4e7003355b950af62', 'Архив отличается от исходного; это новый вход'
    role='val' if STAGE=='calibration' else 'test'
    checkpoints=[4000] if STAGE=='calibration' else [500,1500,4000]
    with np.load(INPUT_PATH,allow_pickle=False) as pack:
        target=pack[f'{role}0_y'].astype(bool)
        inputs={str(ck):pack[f'{role}0_it{ck}'].astype(np.float32) for ck in checkpoints}
    shape=target.shape;assert shape==(125,256,128)
    repeats=5;geometries=[(a,m) for a in [16,32,64] for m in [4,8]]
for p in inputs.values():
    assert p.shape==target.shape and np.isfinite(p).all() and p.min()>=0 and p.max()<=1
full_box=tuple(x for n in shape for x in [0,n])
full=[{'box':full_box,'core':full_box}]
for a,m in geometries:
    cover=windows(shape,a,m)
    assert sum(voxel_count(r['core']) for r in cover)==int(np.prod(shape))
    if MODE=='quick':
        counts=np.zeros(shape,dtype=np.int16)
        for r in cover:counts[tuple(slice(r['core'][i],r['core'][i+1]) for i in [0,2,4])]+=1
        assert np.all(counts==1)
print('Режим:',MODE,'форма:',shape,'повторы:',repeats)
REFERENCE_BOXES = [[0, 20, 0, 20, 0, 20], [0, 20, 0, 20, 12, 36], [0, 20, 0, 20, 28, 52], [0, 20, 0, 20, 44, 68], [0, 20, 0, 20, 60, 84], [0, 20, 0, 20, 76, 100], [0, 20, 0, 20, 92, 116], [0, 20, 0, 20, 108, 128], [0, 20, 12, 36, 0, 20], [0, 20, 12, 36, 12, 36], [0, 20, 12, 36, 28, 52], [0, 20, 12, 36, 44, 68], [0, 20, 12, 36, 60, 84], [0, 20, 12, 36, 76, 100], [0, 20, 12, 36, 92, 116], [0, 20, 12, 36, 108, 128], [0, 20, 28, 52, 0, 20], [0, 20, 28, 52, 12, 36], [0, 20, 28, 52, 28, 52], [0, 20, 28, 52, 44, 68], [0, 20, 28, 52, 60, 84], [0, 20, 28, 52, 76, 100], [0, 20, 28, 52, 92, 116], [0, 20, 28, 52, 108, 128], [0, 20, 28, 68, 12, 36], [0, 20, 44, 68, 0, 20], [0, 20, 44, 68, 12, 36], [0, 20, 44, 68, 28, 52], [0, 20, 44, 68, 44, 68], [0, 20, 44, 68, 60, 84], [0, 20, 44, 68, 76, 100], [0, 20, 44, 68, 92, 116], [0, 20, 44, 68, 108, 128], [0, 20, 60, 84, 0, 20], [0, 20, 60, 84, 12, 36], [0, 20, 60, 84, 28, 52], [0, 20, 60, 84, 44, 68], [0, 20, 60, 84, 60, 84], [0, 20, 60, 84, 76, 100], [0, 20, 60, 84, 92, 116], [0, 20, 60, 84, 108, 128], [0, 20, 76, 100, 0, 20], [0, 20, 76, 100, 12, 36], [0, 20, 76, 100, 28, 52], [0, 20, 76, 100, 44, 68], [0, 20, 76, 100, 60, 84], [0, 20, 76, 100, 76, 100], [0, 20, 76, 100, 92, 116], [0, 20, 76, 100, 108, 128], [0, 20, 92, 116, 0, 20], [0, 20, 92, 116, 12, 36], [0, 20, 92, 116, 28, 52], [0, 20, 92, 116, 44, 68], [0, 20, 92, 116, 60, 84], [0, 20, 92, 116, 76, 100], [0, 20, 92, 116, 92, 116], [0, 20, 92, 116, 108, 128], [0, 20, 108, 132, 0, 20], [0, 20, 108, 132, 12, 36], [0, 20, 108, 132, 28, 52], [0, 20, 108, 132, 44, 68], [0, 20, 108, 132, 60, 84], [0, 20, 108, 132, 76, 100], [0, 20, 108, 132, 92, 116], [0, 20, 108, 132, 108, 128], [0, 20, 124, 148, 0, 20], [0, 20, 124, 148, 12, 36], [0, 20, 124, 148, 28, 52], [0, 20, 124, 148, 44, 68], [0, 20, 124, 148, 60, 84], [0, 20, 124, 148, 76, 100], [0, 20, 124, 148, 92, 116], [0, 20, 124, 148, 108, 128], [0, 20, 140, 164, 0, 20], [0, 20, 140, 164, 12, 36], [0, 20, 140, 164, 28, 52], [0, 20, 140, 164, 44, 68], [0, 20, 140, 164, 60, 84], [0, 20, 140, 164, 76, 100], [0, 20, 140, 164, 92, 116], [0, 20, 140, 164, 108, 128], [0, 20, 156, 180, 0, 20], [0, 20, 156, 180, 12, 36], [0, 20, 156, 180, 28, 52], [0, 20, 156, 180, 44, 68], [0, 20, 156, 180, 60, 84], [0, 20, 156, 180, 76, 100], [0, 20, 156, 180, 92, 116], [0, 20, 156, 180, 108, 128], [0, 20, 172, 196, 0, 20], [0, 20, 172, 196, 12, 36], [0, 20, 172, 196, 28, 52], [0, 20, 172, 196, 44, 68], [0, 20, 172, 196, 60, 84], [0, 20, 172, 196, 76, 100], [0, 20, 172, 196, 92, 116], [0, 20, 172, 196, 108, 128], [0, 20, 188, 212, 0, 20], [0, 20, 188, 212, 12, 36], [0, 20, 188, 212, 28, 52], [0, 20, 188, 212, 44, 68], [0, 20, 188, 212, 60, 84], [0, 20, 188, 212, 76, 100], [0, 20, 188, 212, 92, 116], [0, 20, 188, 212, 108, 128], [0, 20, 204, 228, 0, 20], [0, 20, 204, 228, 12, 36], [0, 20, 204, 228, 28, 52], [0, 20, 204, 228, 44, 68], [0, 20, 204, 228, 60, 84], [0, 20, 204, 228, 76, 100], [0, 20, 204, 228, 92, 116], [0, 20, 204, 228, 108, 128], [0, 20, 220, 244, 0, 20], [0, 20, 220, 244, 12, 36], [0, 20, 220, 244, 28, 52], [0, 20, 220, 244, 44, 68], [0, 20, 220, 244, 60, 84], [0, 20, 220, 244, 76, 100], [0, 20, 220, 244, 92, 116], [0, 20, 220, 244, 108, 128], [0, 20, 236, 256, 0, 20], [0, 20, 236, 256, 12, 36], [0, 20, 236, 256, 28, 52], [0, 20, 236, 256, 44, 68], [0, 20, 236, 256, 60, 84], [0, 20, 236, 256, 76, 100], [0, 20, 236, 256, 92, 116], [0, 20, 236, 256, 108, 128], [0, 24, 0, 24, 0, 24], [0, 24, 0, 24, 0, 40], [0, 24, 0, 24, 8, 40], [0, 24, 0, 24, 24, 56], [0, 24, 0, 24, 40, 72], [0, 24, 0, 24, 56, 88], [0, 24, 0, 24, 72, 104], [0, 24, 0, 24, 88, 120], [0, 24, 0, 24, 104, 128], [0, 24, 8, 40, 0, 24], [0, 24, 8, 40, 8, 40], [0, 24, 8, 40, 24, 56], [0, 24, 8, 40, 40, 72], [0, 24, 8, 40, 56, 88], [0, 24, 8, 40, 72, 104], [0, 24, 8, 40, 88, 120], [0, 24, 8, 40, 104, 128], [0, 24, 24, 56, 0, 24], [0, 24, 24, 56, 8, 40], [0, 24, 24, 56, 24, 56], [0, 24, 24, 56, 40, 72], [0, 24, 24, 56, 56, 88], [0, 24, 24, 56, 72, 104], [0, 24, 24, 56, 88, 120], [0, 24, 24, 56, 104, 128], [0, 24, 24, 72, 40, 72], [0, 24, 40, 72, 0, 24], [0, 24, 40, 72, 8, 40], [0, 24, 40, 72, 24, 56], [0, 24, 40, 72, 40, 72], [0, 24, 40, 72, 56, 88], [0, 24, 40, 72, 72, 104], [0, 24, 40, 72, 88, 120], [0, 24, 40, 72, 104, 128], [0, 24, 56, 88, 0, 24], [0, 24, 56, 88, 8, 40], [0, 24, 56, 88, 24, 56], [0, 24, 56, 88, 40, 72], [0, 24, 56, 88, 56, 88], [0, 24, 56, 88, 72, 104], [0, 24, 56, 88, 88, 120], [0, 24, 56, 88, 104, 128], [0, 24, 72, 104, 0, 24], [0, 24, 72, 104, 8, 40], [0, 24, 72, 104, 24, 56], [0, 24, 72, 104, 40, 72], [0, 24, 72, 104, 56, 88], [0, 24, 72, 104, 72, 104], [0, 24, 72, 104, 88, 120], [0, 24, 72, 104, 104, 128], [0, 24, 88, 120, 0, 24], [0, 24, 88, 120, 8, 40], [0, 24, 88, 120, 24, 56], [0, 24, 88, 120, 40, 72], [0, 24, 88, 120, 56, 88], [0, 24, 88, 120, 72, 104], [0, 24, 88, 120, 88, 120], [0, 24, 88, 120, 104, 128], [0, 24, 88, 136, 56, 104], [0, 24, 104, 136, 0, 24], [0, 24, 104, 136, 8, 40], [0, 24, 104, 136, 24, 56], [0, 24, 104, 136, 40, 72], [0, 24, 104, 136, 56, 88], [0, 24, 104, 136, 72, 104], [0, 24, 104, 136, 88, 120], [0, 24, 104, 136, 104, 128], [0, 24, 120, 152, 0, 24], [0, 24, 120, 152, 8, 40], [0, 24, 120, 152, 24, 56], [0, 24, 120, 152, 40, 72], [0, 24, 120, 152, 56, 88], [0, 24, 120, 152, 72, 104], [0, 24, 120, 152, 88, 120], [0, 24, 120, 152, 104, 128], [0, 24, 136, 168, 0, 24], [0, 24, 136, 168, 8, 40], [0, 24, 136, 168, 24, 56], [0, 24, 136, 168, 40, 72], [0, 24, 136, 168, 56, 88], [0, 24, 136, 168, 72, 104], [0, 24, 136, 168, 88, 120], [0, 24, 136, 168, 104, 128], [0, 24, 152, 184, 0, 24], [0, 24, 152, 184, 8, 40], [0, 24, 152, 184, 24, 56], [0, 24, 152, 184, 40, 72], [0, 24, 152, 184, 56, 88], [0, 24, 152, 184, 72, 104], [0, 24, 152, 184, 88, 120], [0, 24, 152, 184, 104, 128], [0, 24, 168, 200, 0, 24], [0, 24, 168, 200, 8, 40], [0, 24, 168, 200, 24, 56], [0, 24, 168, 200, 40, 72], [0, 24, 168, 200, 56, 88], [0, 24, 168, 200, 72, 104], [0, 24, 168, 200, 88, 120], [0, 24, 168, 200, 104, 128], [0, 24, 184, 216, 0, 24], [0, 24, 184, 216, 8, 40], [0, 24, 184, 216, 24, 56], [0, 24, 184, 216, 40, 72], [0, 24, 184, 216, 56, 88], [0, 24, 184, 216, 72, 104], [0, 24, 184, 216, 88, 120], [0, 24, 184, 216, 104, 128], [0, 24, 200, 232, 0, 24], [0, 24, 200, 232, 8, 40], [0, 24, 200, 232, 24, 56], [0, 24, 200, 232, 40, 72], [0, 24, 200, 232, 56, 88], [0, 24, 200, 232, 72, 104], [0, 24, 200, 232, 88, 120], [0, 24, 200, 232, 104, 128], [0, 24, 216, 248, 0, 24], [0, 24, 216, 248, 8, 40], [0, 24, 216, 248, 24, 56], [0, 24, 216, 248, 40, 72], [0, 24, 216, 248, 56, 88], [0, 24, 216, 248, 72, 104], [0, 24, 216, 248, 88, 120], [0, 24, 216, 248, 104, 128], [0, 24, 232, 256, 0, 24], [0, 24, 232, 256, 8, 40], [0, 24, 232, 256, 24, 56], [0, 24, 232, 256, 40, 72], [0, 24, 232, 256, 56, 88], [0, 24, 232, 256, 72, 104], [0, 24, 232, 256, 88, 120], [0, 24, 232, 256, 104, 128], [0, 36, 0, 20, 76, 100], [0, 36, 0, 36, 0, 36], [0, 36, 0, 36, 28, 68], [0, 36, 0, 36, 60, 100], [0, 36, 0, 36, 92, 128], [0, 36, 12, 36, 44, 68], [0, 36, 12, 36, 92, 128], [0, 36, 28, 52, 60, 84], [0, 36, 28, 68, 0, 36], [0, 36, 28, 68, 12, 36], [0, 36, 28, 68, 28, 68], [0, 36, 28, 68, 60, 100], [0, 36, 28, 68, 92, 116], [0, 36, 28, 68, 92, 128], [0, 36, 60, 84, 0, 36], [0, 36, 60, 84, 28, 68], [0, 36, 60, 100, 0, 36], [0, 36, 60, 100, 28, 68], [0, 36, 60, 100, 60, 100], [0, 36, 60, 100, 92, 116], [0, 36, 60, 100, 92, 128], [0, 36, 92, 116, 28, 68], [0, 36, 92, 116, 108, 128], [0, 36, 92, 132, 0, 36], [0, 36, 92, 132, 12, 36], [0, 36, 92, 132, 28, 68], [0, 36, 92, 132, 60, 100], [0, 36, 92, 132, 92, 128], [0, 36, 108, 132, 12, 36], [0, 36, 124, 164, 0, 36], [0, 36, 124, 164, 28, 68], [0, 36, 124, 164, 60, 100], [0, 36, 124, 164, 92, 128], [0, 36, 156, 196, 0, 36], [0, 36, 156, 196, 28, 68], [0, 36, 156, 196, 60, 100], [0, 36, 156, 196, 92, 128], [0, 36, 188, 212, 44, 68], [0, 36, 188, 228, 0, 36], [0, 36, 188, 228, 0, 68], [0, 36, 188, 228, 28, 68], [0, 36, 188, 228, 60, 100], [0, 36, 188, 228, 92, 128], [0, 36, 188, 228, 108, 128], [0, 36, 220, 256, 0, 36], [0, 36, 220, 256, 28, 68], [0, 36, 220, 256, 60, 100], [0, 36, 220, 256, 92, 128], [0, 36, 236, 256, 12, 36], [0, 40, 0, 24, 0, 40], [0, 40, 0, 40, 0, 40], [0, 40, 0, 40, 24, 72], [0, 40, 0, 40, 40, 72], [0, 40, 0, 40, 56, 104], [0, 40, 0, 40, 88, 128], [0, 40, 8, 40, 40, 72], [0, 40, 8, 40, 56, 104], [0, 40, 24, 56, 88, 120], [0, 40, 24, 56, 88, 128], [0, 40, 24, 72, 0, 40], [0, 40, 24, 72, 8, 40], [0, 40, 24, 72, 24, 72], [0, 40, 24, 72, 40, 72], [0, 40, 24, 72, 56, 104], [0, 40, 24, 72, 88, 128], [0, 40, 40, 72, 8, 40], [0, 40, 56, 104, 0, 40], [0, 40, 56, 104, 24, 72], [0, 40, 56, 104, 56, 104], [0, 40, 56, 104, 88, 128], [0, 40, 72, 104, 0, 40], [0, 40, 72, 104, 8, 40], [0, 40, 88, 136, 0, 40], [0, 40, 88, 136, 24, 56], [0, 40, 88, 136, 24, 72], [0, 40, 88, 136, 56, 104], [0, 40, 88, 136, 88, 128], [0, 40, 120, 168, 0, 40], [0, 40, 120, 168, 24, 72], [0, 40, 120, 168, 56, 88], [0, 40, 120, 168, 56, 104], [0, 40, 120, 168, 88, 120], [0, 40, 120, 168, 88, 128], [0, 40, 152, 200, 0, 40], [0, 40, 152, 200, 24, 72], [0, 40, 152, 200, 56, 104], [0, 40, 152, 200, 88, 128], [0, 40, 184, 216, 72, 104], [0, 40, 184, 232, 0, 24], [0, 40, 184, 232, 0, 40], [0, 40, 184, 232, 24, 72], [0, 40, 184, 232, 56, 104], [0, 40, 184, 232, 88, 128], [0, 40, 216, 256, 0, 40], [0, 40, 216, 256, 24, 56], [0, 40, 216, 256, 24, 72], [0, 40, 216, 256, 56, 104], [0, 40, 216, 256, 88, 128], [0, 40, 216, 256, 104, 128], [0, 40, 232, 256, 8, 40], [0, 52, 28, 68, 12, 68], [0, 52, 60, 116, 0, 68], [0, 52, 124, 196, 28, 68], [0, 52, 188, 228, 0, 68], [0, 52, 188, 256, 0, 68], [0, 56, 8, 72, 0, 72], [0, 68, 0, 68, 0, 68], [0, 68, 0, 68, 60, 100], [0, 68, 0, 68, 60, 116], [0, 68, 0, 68, 60, 128], [0, 68, 60, 116, 60, 128], [0, 68, 60, 132, 0, 68], [0, 68, 60, 132, 60, 128], [0, 68, 124, 196, 0, 68], [0, 68, 124, 196, 28, 68], [0, 68, 124, 196, 60, 128], [0, 68, 188, 228, 0, 68], [0, 68, 188, 228, 28, 68], [0, 68, 188, 256, 0, 68], [0, 68, 188, 256, 60, 128], [0, 68, 220, 256, 60, 128], [0, 72, 0, 72, 0, 72], [0, 72, 0, 72, 56, 128], [0, 72, 8, 72, 56, 120], [0, 72, 8, 72, 56, 128], [0, 72, 56, 104, 56, 104], [0, 72, 56, 136, 0, 72], [0, 72, 56, 136, 56, 104], [0, 72, 56, 136, 56, 128], [0, 72, 120, 200, 0, 72], [0, 72, 120, 200, 8, 72], [0, 72, 120, 200, 24, 72], [0, 72, 120, 200, 56, 120], [0, 72, 120, 200, 56, 128], [0, 72, 184, 232, 0, 72], [0, 72, 184, 256, 0, 72], [0, 72, 184, 256, 56, 104], [0, 72, 184, 256, 56, 128], [0, 125, 0, 256, 0, 128], [8, 40, 0, 24, 0, 24], [8, 40, 0, 24, 8, 40], [8, 40, 0, 24, 24, 56], [8, 40, 0, 24, 40, 72], [8, 40, 0, 24, 56, 88], [8, 40, 0, 24, 72, 104], [8, 40, 0, 24, 88, 120], [8, 40, 0, 24, 104, 128], [8, 40, 8, 40, 0, 24], [8, 40, 8, 40, 8, 40], [8, 40, 8, 40, 24, 56], [8, 40, 8, 40, 40, 72], [8, 40, 8, 40, 56, 88], [8, 40, 8, 40, 72, 104], [8, 40, 8, 40, 88, 120], [8, 40, 8, 40, 88, 128], [8, 40, 8, 40, 104, 128], [8, 40, 24, 56, 0, 24], [8, 40, 24, 56, 8, 40], [8, 40, 24, 56, 24, 56], [8, 40, 24, 56, 40, 72], [8, 40, 24, 56, 56, 88], [8, 40, 24, 56, 72, 104], [8, 40, 24, 56, 88, 120], [8, 40, 24, 56, 104, 128], [8, 40, 40, 72, 0, 24], [8, 40, 40, 72, 8, 40], [8, 40, 40, 72, 24, 56], [8, 40, 40, 72, 40, 72], [8, 40, 40, 72, 56, 88], [8, 40, 40, 72, 72, 104], [8, 40, 40, 72, 88, 120], [8, 40, 40, 72, 104, 128], [8, 40, 56, 88, 0, 24], [8, 40, 56, 88, 8, 40], [8, 40, 56, 88, 24, 56], [8, 40, 56, 88, 40, 72], [8, 40, 56, 88, 56, 88], [8, 40, 56, 88, 56, 104], [8, 40, 56, 88, 72, 104], [8, 40, 56, 88, 88, 120], [8, 40, 56, 88, 104, 128], [8, 40, 56, 104, 40, 72], [8, 40, 56, 104, 88, 128], [8, 40, 56, 104, 104, 128], [8, 40, 72, 104, 0, 24], [8, 40, 72, 104, 8, 40], [8, 40, 72, 104, 24, 56], [8, 40, 72, 104, 40, 72], [8, 40, 72, 104, 56, 88], [8, 40, 72, 104, 72, 104], [8, 40, 72, 104, 88, 120], [8, 40, 72, 104, 104, 128], [8, 40, 88, 120, 0, 24], [8, 40, 88, 120, 8, 40], [8, 40, 88, 120, 24, 56], [8, 40, 88, 120, 40, 72], [8, 40, 88, 120, 56, 88], [8, 40, 88, 120, 72, 104], [8, 40, 88, 120, 88, 120], [8, 40, 88, 120, 104, 128], [8, 40, 104, 136, 0, 24], [8, 40, 104, 136, 8, 40], [8, 40, 104, 136, 24, 56], [8, 40, 104, 136, 40, 72], [8, 40, 104, 136, 56, 88], [8, 40, 104, 136, 72, 104], [8, 40, 104, 136, 88, 120], [8, 40, 104, 136, 104, 128], [8, 40, 120, 152, 0, 24], [8, 40, 120, 152, 8, 40], [8, 40, 120, 152, 24, 56], [8, 40, 120, 152, 40, 72], [8, 40, 120, 152, 56, 88], [8, 40, 120, 152, 72, 104], [8, 40, 120, 152, 88, 120], [8, 40, 120, 152, 104, 128], [8, 40, 136, 168, 0, 24], [8, 40, 136, 168, 0, 40], [8, 40, 136, 168, 8, 40], [8, 40, 136, 168, 24, 56], [8, 40, 136, 168, 40, 72], [8, 40, 136, 168, 56, 88], [8, 40, 136, 168, 72, 104], [8, 40, 136, 168, 88, 120], [8, 40, 136, 168, 104, 128], [8, 40, 152, 184, 0, 24], [8, 40, 152, 184, 0, 40], [8, 40, 152, 184, 8, 40], [8, 40, 152, 184, 24, 56], [8, 40, 152, 184, 40, 72], [8, 40, 152, 184, 56, 88], [8, 40, 152, 184, 72, 104], [8, 40, 152, 184, 88, 120], [8, 40, 152, 184, 104, 128], [8, 40, 152, 200, 88, 128], [8, 40, 168, 200, 0, 24], [8, 40, 168, 200, 8, 40], [8, 40, 168, 200, 24, 56], [8, 40, 168, 200, 40, 72], [8, 40, 168, 200, 56, 88], [8, 40, 168, 200, 72, 104], [8, 40, 168, 200, 88, 120], [8, 40, 168, 200, 104, 128], [8, 40, 184, 216, 0, 24], [8, 40, 184, 216, 8, 40], [8, 40, 184, 216, 24, 56], [8, 40, 184, 216, 40, 72], [8, 40, 184, 216, 56, 88], [8, 40, 184, 216, 72, 104], [8, 40, 184, 216, 88, 120], [8, 40, 184, 216, 104, 128], [8, 40, 184, 232, 88, 128], [8, 40, 200, 232, 0, 24], [8, 40, 200, 232, 8, 40], [8, 40, 200, 232, 24, 56], [8, 40, 200, 232, 40, 72], [8, 40, 200, 232, 56, 88], [8, 40, 200, 232, 72, 104], [8, 40, 200, 232, 88, 120], [8, 40, 200, 232, 104, 128], [8, 40, 216, 248, 0, 24], [8, 40, 216, 248, 8, 40], [8, 40, 216, 248, 24, 56], [8, 40, 216, 248, 40, 72], [8, 40, 216, 248, 56, 88], [8, 40, 216, 248, 72, 104], [8, 40, 216, 248, 88, 120], [8, 40, 216, 248, 104, 128], [8, 40, 232, 256, 0, 24], [8, 40, 232, 256, 8, 40], [8, 40, 232, 256, 24, 56], [8, 40, 232, 256, 40, 72], [8, 40, 232, 256, 56, 88], [8, 40, 232, 256, 72, 104], [8, 40, 232, 256, 88, 120], [8, 40, 232, 256, 104, 128], [12, 36, 0, 20, 0, 20], [12, 36, 0, 20, 12, 36], [12, 36, 0, 20, 28, 52], [12, 36, 0, 20, 44, 68], [12, 36, 0, 20, 60, 84], [12, 36, 0, 20, 76, 100], [12, 36, 0, 20, 92, 116], [12, 36, 0, 20, 108, 128], [12, 36, 12, 36, 0, 20], [12, 36, 12, 36, 12, 36], [12, 36, 12, 36, 28, 52], [12, 36, 12, 36, 44, 68], [12, 36, 12, 36, 60, 84], [12, 36, 12, 36, 76, 100], [12, 36, 12, 36, 92, 116], [12, 36, 12, 36, 108, 128], [12, 36, 28, 52, 0, 20], [12, 36, 28, 52, 12, 36], [12, 36, 28, 52, 28, 52], [12, 36, 28, 52, 44, 68], [12, 36, 28, 52, 60, 84], [12, 36, 28, 52, 76, 100], [12, 36, 28, 52, 92, 116], [12, 36, 28, 52, 108, 128], [12, 36, 44, 68, 0, 20], [12, 36, 44, 68, 12, 36], [12, 36, 44, 68, 28, 52], [12, 36, 44, 68, 44, 68], [12, 36, 44, 68, 60, 84], [12, 36, 44, 68, 76, 100], [12, 36, 44, 68, 92, 116], [12, 36, 44, 68, 108, 128], [12, 36, 60, 84, 0, 20], [12, 36, 60, 84, 12, 36], [12, 36, 60, 84, 28, 52], [12, 36, 60, 84, 44, 68], [12, 36, 60, 84, 60, 84], [12, 36, 60, 84, 76, 100], [12, 36, 60, 84, 92, 116], [12, 36, 60, 84, 108, 128], [12, 36, 76, 100, 0, 20], [12, 36, 76, 100, 12, 36], [12, 36, 76, 100, 28, 52], [12, 36, 76, 100, 44, 68], [12, 36, 76, 100, 60, 84], [12, 36, 76, 100, 76, 100], [12, 36, 76, 100, 92, 116], [12, 36, 76, 100, 108, 128], [12, 36, 92, 116, 0, 20], [12, 36, 92, 116, 12, 36], [12, 36, 92, 116, 28, 52], [12, 36, 92, 116, 44, 68], [12, 36, 92, 116, 60, 84], [12, 36, 92, 116, 76, 100], [12, 36, 92, 116, 92, 116], [12, 36, 92, 116, 108, 128], [12, 36, 92, 132, 60, 100], [12, 36, 108, 132, 0, 20], [12, 36, 108, 132, 12, 36], [12, 36, 108, 132, 28, 52], [12, 36, 108, 132, 44, 68], [12, 36, 108, 132, 60, 84], [12, 36, 108, 132, 76, 100], [12, 36, 108, 132, 92, 116], [12, 36, 108, 132, 108, 128], [12, 36, 124, 148, 0, 20], [12, 36, 124, 148, 12, 36], [12, 36, 124, 148, 28, 52], [12, 36, 124, 148, 44, 68], [12, 36, 124, 148, 60, 84], [12, 36, 124, 148, 76, 100], [12, 36, 124, 148, 92, 116], [12, 36, 124, 148, 108, 128], [12, 36, 124, 164, 28, 68], [12, 36, 140, 164, 0, 20], [12, 36, 140, 164, 0, 36], [12, 36, 140, 164, 12, 36], [12, 36, 140, 164, 28, 52], [12, 36, 140, 164, 44, 68], [12, 36, 140, 164, 60, 84], [12, 36, 140, 164, 76, 100], [12, 36, 140, 164, 92, 116], [12, 36, 140, 164, 92, 128], [12, 36, 140, 164, 108, 128], [12, 36, 156, 180, 0, 20], [12, 36, 156, 180, 0, 36], [12, 36, 156, 180, 12, 36], [12, 36, 156, 180, 28, 52], [12, 36, 156, 180, 44, 68], [12, 36, 156, 180, 60, 84], [12, 36, 156, 180, 76, 100], [12, 36, 156, 180, 92, 116], [12, 36, 156, 180, 108, 128], [12, 36, 156, 196, 0, 36], [12, 36, 172, 196, 0, 20], [12, 36, 172, 196, 12, 36], [12, 36, 172, 196, 28, 52], [12, 36, 172, 196, 44, 68], [12, 36, 172, 196, 60, 84], [12, 36, 172, 196, 76, 100], [12, 36, 172, 196, 92, 116], [12, 36, 172, 196, 108, 128], [12, 36, 188, 212, 0, 20], [12, 36, 188, 212, 12, 36], [12, 36, 188, 212, 28, 52], [12, 36, 188, 212, 44, 68], [12, 36, 188, 212, 60, 84], [12, 36, 188, 212, 76, 100], [12, 36, 188, 212, 92, 116], [12, 36, 188, 212, 108, 128], [12, 36, 188, 228, 60, 100], [12, 36, 204, 228, 0, 20], [12, 36, 204, 228, 12, 36], [12, 36, 204, 228, 28, 52], [12, 36, 204, 228, 44, 68], [12, 36, 204, 228, 60, 84], [12, 36, 204, 228, 76, 100], [12, 36, 204, 228, 92, 116], [12, 36, 204, 228, 108, 128], [12, 36, 220, 244, 0, 20], [12, 36, 220, 244, 12, 36], [12, 36, 220, 244, 28, 52], [12, 36, 220, 244, 44, 68], [12, 36, 220, 244, 60, 84], [12, 36, 220, 244, 76, 100], [12, 36, 220, 244, 92, 116], [12, 36, 220, 244, 108, 128], [12, 36, 236, 256, 0, 20], [12, 36, 236, 256, 12, 36], [12, 36, 236, 256, 28, 52], [12, 36, 236, 256, 44, 68], [12, 36, 236, 256, 60, 84], [12, 36, 236, 256, 76, 100], [12, 36, 236, 256, 92, 116], [12, 36, 236, 256, 108, 128], [12, 68, 140, 196, 60, 128], [24, 56, 0, 24, 0, 24], [24, 56, 0, 24, 8, 40], [24, 56, 0, 24, 24, 56], [24, 56, 0, 24, 40, 72], [24, 56, 0, 24, 56, 88], [24, 56, 0, 24, 72, 104], [24, 56, 0, 24, 88, 120], [24, 56, 0, 24, 104, 128], [24, 56, 8, 40, 0, 24], [24, 56, 8, 40, 0, 40], [24, 56, 8, 40, 8, 40], [24, 56, 8, 40, 24, 56], [24, 56, 8, 40, 40, 72], [24, 56, 8, 40, 56, 88], [24, 56, 8, 40, 72, 104], [24, 56, 8, 40, 88, 120], [24, 56, 8, 40, 88, 128], [24, 56, 8, 40, 104, 128], [24, 56, 24, 56, 0, 24], [24, 56, 24, 56, 8, 40], [24, 56, 24, 56, 24, 56], [24, 56, 24, 56, 40, 72], [24, 56, 24, 56, 56, 88], [24, 56, 24, 56, 72, 104], [24, 56, 24, 56, 88, 120], [24, 56, 24, 56, 104, 128], [24, 56, 24, 72, 0, 24], [24, 56, 24, 72, 0, 40], [24, 56, 24, 72, 88, 120], [24, 56, 24, 72, 88, 128], [24, 56, 40, 72, 0, 24], [24, 56, 40, 72, 8, 40], [24, 56, 40, 72, 24, 56], [24, 56, 40, 72, 40, 72], [24, 56, 40, 72, 56, 88], [24, 56, 40, 72, 72, 104], [24, 56, 40, 72, 88, 120], [24, 56, 40, 72, 104, 128], [24, 56, 56, 88, 0, 24], [24, 56, 56, 88, 8, 40], [24, 56, 56, 88, 24, 56], [24, 56, 56, 88, 40, 72], [24, 56, 56, 88, 56, 88], [24, 56, 56, 88, 72, 104], [24, 56, 56, 88, 88, 120], [24, 56, 56, 88, 104, 128], [24, 56, 56, 104, 0, 40], [24, 56, 56, 104, 56, 104], [24, 56, 72, 104, 0, 24], [24, 56, 72, 104, 8, 40], [24, 56, 72, 104, 24, 56], [24, 56, 72, 104, 40, 72], [24, 56, 72, 104, 56, 88], [24, 56, 72, 104, 72, 104], [24, 56, 72, 104, 88, 120], [24, 56, 72, 104, 104, 128], [24, 56, 88, 120, 0, 24], [24, 56, 88, 120, 8, 40], [24, 56, 88, 120, 24, 56], [24, 56, 88, 120, 40, 72], [24, 56, 88, 120, 56, 88], [24, 56, 88, 120, 72, 104], [24, 56, 88, 120, 88, 120], [24, 56, 88, 120, 104, 128], [24, 56, 104, 136, 0, 24], [24, 56, 104, 136, 8, 40], [24, 56, 104, 136, 24, 56], [24, 56, 104, 136, 40, 72], [24, 56, 104, 136, 56, 88], [24, 56, 104, 136, 72, 104], [24, 56, 104, 136, 88, 120], [24, 56, 104, 136, 104, 128], [24, 56, 120, 152, 0, 24], [24, 56, 120, 152, 8, 40], [24, 56, 120, 152, 24, 56], [24, 56, 120, 152, 40, 72], [24, 56, 120, 152, 56, 88], [24, 56, 120, 152, 72, 104], [24, 56, 120, 152, 88, 120], [24, 56, 120, 152, 104, 128], [24, 56, 136, 168, 0, 24], [24, 56, 136, 168, 8, 40], [24, 56, 136, 168, 24, 56], [24, 56, 136, 168, 40, 72], [24, 56, 136, 168, 56, 88], [24, 56, 136, 168, 72, 104], [24, 56, 136, 168, 88, 120], [24, 56, 136, 168, 104, 128], [24, 56, 152, 184, 0, 24], [24, 56, 152, 184, 0, 40], [24, 56, 152, 184, 8, 40], [24, 56, 152, 184, 24, 56], [24, 56, 152, 184, 40, 72], [24, 56, 152, 184, 56, 88], [24, 56, 152, 184, 72, 104], [24, 56, 152, 184, 88, 120], [24, 56, 152, 184, 104, 128], [24, 56, 152, 200, 0, 40], [24, 56, 152, 200, 24, 56], [24, 56, 168, 200, 0, 24], [24, 56, 168, 200, 8, 40], [24, 56, 168, 200, 24, 56], [24, 56, 168, 200, 40, 72], [24, 56, 168, 200, 56, 88], [24, 56, 168, 200, 72, 104], [24, 56, 168, 200, 88, 120], [24, 56, 168, 200, 104, 128], [24, 56, 184, 216, 0, 24], [24, 56, 184, 216, 8, 40], [24, 56, 184, 216, 24, 56], [24, 56, 184, 216, 40, 72], [24, 56, 184, 216, 56, 88], [24, 56, 184, 216, 72, 104], [24, 56, 184, 216, 88, 120], [24, 56, 184, 216, 104, 128], [24, 56, 200, 232, 0, 24], [24, 56, 200, 232, 8, 40], [24, 56, 200, 232, 24, 56], [24, 56, 200, 232, 40, 72], [24, 56, 200, 232, 56, 88], [24, 56, 200, 232, 72, 104], [24, 56, 200, 232, 88, 120], [24, 56, 200, 232, 104, 128], [24, 56, 216, 248, 0, 24], [24, 56, 216, 248, 8, 40], [24, 56, 216, 248, 24, 56], [24, 56, 216, 248, 24, 72], [24, 56, 216, 248, 40, 72], [24, 56, 216, 248, 56, 88], [24, 56, 216, 248, 72, 104], [24, 56, 216, 248, 88, 120], [24, 56, 216, 248, 104, 128], [24, 56, 216, 256, 24, 72], [24, 56, 216, 256, 104, 128], [24, 56, 232, 256, 0, 24], [24, 56, 232, 256, 8, 40], [24, 56, 232, 256, 24, 56], [24, 56, 232, 256, 40, 72], [24, 56, 232, 256, 56, 88], [24, 56, 232, 256, 72, 104], [24, 56, 232, 256, 88, 120], [24, 56, 232, 256, 104, 128], [24, 72, 0, 40, 0, 40], [24, 72, 0, 40, 24, 72], [24, 72, 0, 40, 56, 104], [24, 72, 0, 40, 88, 128], [24, 72, 8, 40, 56, 88], [24, 72, 8, 40, 88, 120], [24, 72, 24, 72, 0, 40], [24, 72, 24, 72, 24, 72], [24, 72, 24, 72, 56, 104], [24, 72, 24, 72, 88, 128], [24, 72, 56, 88, 104, 128], [24, 72, 56, 104, 0, 40], [24, 72, 56, 104, 24, 72], [24, 72, 56, 104, 56, 104], [24, 72, 56, 104, 88, 128], [24, 72, 72, 104, 24, 72], [24, 72, 88, 136, 0, 40], [24, 72, 88, 136, 8, 40], [24, 72, 88, 136, 24, 72], [24, 72, 88, 136, 56, 104], [24, 72, 88, 136, 88, 128], [24, 72, 104, 136, 72, 104], [24, 72, 120, 168, 0, 40], [24, 72, 120, 168, 24, 72], [24, 72, 120, 168, 56, 104], [24, 72, 120, 168, 88, 120], [24, 72, 120, 168, 88, 128], [24, 72, 136, 168, 0, 40], [24, 72, 136, 168, 8, 40], [24, 72, 136, 168, 24, 72], [24, 72, 152, 184, 56, 104], [24, 72, 152, 200, 0, 40], [24, 72, 152, 200, 24, 72], [24, 72, 152, 200, 56, 104], [24, 72, 152, 200, 88, 128], [24, 72, 184, 216, 40, 72], [24, 72, 184, 232, 0, 40], [24, 72, 184, 232, 0, 72], [24, 72, 184, 232, 24, 72], [24, 72, 184, 232, 56, 104], [24, 72, 184, 232, 88, 128], [24, 72, 216, 256, 0, 40], [24, 72, 216, 256, 24, 72], [24, 72, 216, 256, 56, 104], [24, 72, 216, 256, 88, 128], [28, 52, 0, 20, 0, 20], [28, 52, 0, 20, 12, 36], [28, 52, 0, 20, 28, 52], [28, 52, 0, 20, 44, 68], [28, 52, 0, 20, 60, 84], [28, 52, 0, 20, 76, 100], [28, 52, 0, 20, 92, 116], [28, 52, 0, 20, 108, 128], [28, 52, 12, 36, 0, 20], [28, 52, 12, 36, 12, 36], [28, 52, 12, 36, 28, 52], [28, 52, 12, 36, 44, 68], [28, 52, 12, 36, 60, 84], [28, 52, 12, 36, 76, 100], [28, 52, 12, 36, 92, 116], [28, 52, 12, 36, 108, 128], [28, 52, 28, 52, 0, 20], [28, 52, 28, 52, 12, 36], [28, 52, 28, 52, 28, 52], [28, 52, 28, 52, 44, 68], [28, 52, 28, 52, 60, 84], [28, 52, 28, 52, 76, 100], [28, 52, 28, 52, 92, 116], [28, 52, 28, 52, 108, 128], [28, 52, 28, 68, 28, 68], [28, 52, 44, 68, 0, 20], [28, 52, 44, 68, 12, 36], [28, 52, 44, 68, 28, 52], [28, 52, 44, 68, 44, 68], [28, 52, 44, 68, 60, 84], [28, 52, 44, 68, 76, 100], [28, 52, 44, 68, 92, 116], [28, 52, 44, 68, 108, 128], [28, 52, 60, 84, 0, 20], [28, 52, 60, 84, 12, 36], [28, 52, 60, 84, 28, 52], [28, 52, 60, 84, 44, 68], [28, 52, 60, 84, 60, 84], [28, 52, 60, 84, 76, 100], [28, 52, 60, 84, 92, 116], [28, 52, 60, 84, 108, 128], [28, 52, 76, 100, 0, 20], [28, 52, 76, 100, 12, 36], [28, 52, 76, 100, 28, 52], [28, 52, 76, 100, 44, 68], [28, 52, 76, 100, 60, 84], [28, 52, 76, 100, 76, 100], [28, 52, 76, 100, 92, 116], [28, 52, 76, 100, 108, 128], [28, 52, 92, 116, 0, 20], [28, 52, 92, 116, 12, 36], [28, 52, 92, 116, 28, 52], [28, 52, 92, 116, 44, 68], [28, 52, 92, 116, 60, 84], [28, 52, 92, 116, 76, 100], [28, 52, 92, 116, 92, 116], [28, 52, 92, 116, 108, 128], [28, 52, 92, 132, 60, 100], [28, 52, 108, 132, 0, 20], [28, 52, 108, 132, 12, 36], [28, 52, 108, 132, 28, 52], [28, 52, 108, 132, 44, 68], [28, 52, 108, 132, 60, 84], [28, 52, 108, 132, 76, 100], [28, 52, 108, 132, 92, 116], [28, 52, 108, 132, 108, 128], [28, 52, 124, 148, 0, 20], [28, 52, 124, 148, 12, 36], [28, 52, 124, 148, 28, 52], [28, 52, 124, 148, 44, 68], [28, 52, 124, 148, 60, 84], [28, 52, 124, 148, 76, 100], [28, 52, 124, 148, 92, 116], [28, 52, 124, 148, 108, 128], [28, 52, 124, 164, 92, 128], [28, 52, 140, 164, 0, 20], [28, 52, 140, 164, 0, 36], [28, 52, 140, 164, 12, 36], [28, 52, 140, 164, 28, 52], [28, 52, 140, 164, 44, 68], [28, 52, 140, 164, 60, 84], [28, 52, 140, 164, 76, 100], [28, 52, 140, 164, 92, 116], [28, 52, 140, 164, 108, 128], [28, 52, 156, 180, 0, 20], [28, 52, 156, 180, 0, 36], [28, 52, 156, 180, 12, 36], [28, 52, 156, 180, 28, 52], [28, 52, 156, 180, 44, 68], [28, 52, 156, 180, 60, 84], [28, 52, 156, 180, 76, 100], [28, 52, 156, 180, 92, 116], [28, 52, 156, 180, 108, 128], [28, 52, 156, 196, 28, 68], [28, 52, 172, 196, 0, 20], [28, 52, 172, 196, 12, 36], [28, 52, 172, 196, 28, 52], [28, 52, 172, 196, 44, 68], [28, 52, 172, 196, 60, 84], [28, 52, 172, 196, 76, 100], [28, 52, 172, 196, 92, 116], [28, 52, 172, 196, 108, 128], [28, 52, 188, 212, 0, 20], [28, 52, 188, 212, 12, 36], [28, 52, 188, 212, 28, 52], [28, 52, 188, 212, 44, 68], [28, 52, 188, 212, 60, 84], [28, 52, 188, 212, 60, 100], [28, 52, 188, 212, 76, 100], [28, 52, 188, 212, 92, 116], [28, 52, 188, 212, 108, 128], [28, 52, 188, 228, 12, 36], [28, 52, 188, 228, 28, 68], [28, 52, 188, 228, 44, 68], [28, 52, 204, 228, 0, 20], [28, 52, 204, 228, 12, 36], [28, 52, 204, 228, 28, 52], [28, 52, 204, 228, 44, 68], [28, 52, 204, 228, 60, 84], [28, 52, 204, 228, 76, 100], [28, 52, 204, 228, 92, 116], [28, 52, 204, 228, 108, 128], [28, 52, 220, 244, 0, 20], [28, 52, 220, 244, 12, 36], [28, 52, 220, 244, 28, 52], [28, 52, 220, 244, 44, 68], [28, 52, 220, 244, 60, 84], [28, 52, 220, 244, 76, 100], [28, 52, 220, 244, 92, 116], [28, 52, 220, 244, 108, 128], [28, 52, 236, 256, 0, 20], [28, 52, 236, 256, 12, 36], [28, 52, 236, 256, 28, 52], [28, 52, 236, 256, 28, 68], [28, 52, 236, 256, 44, 68], [28, 52, 236, 256, 60, 84], [28, 52, 236, 256, 76, 100], [28, 52, 236, 256, 92, 116], [28, 52, 236, 256, 108, 128], [28, 68, 0, 20, 12, 36], [28, 68, 0, 36, 0, 36], [28, 68, 0, 36, 12, 36], [28, 68, 0, 36, 28, 68], [28, 68, 0, 36, 60, 100], [28, 68, 0, 36, 92, 128], [28, 68, 12, 36, 92, 128], [28, 68, 28, 68, 0, 36], [28, 68, 28, 68, 28, 68], [28, 68, 28, 68, 60, 100], [28, 68, 28, 68, 92, 128], [28, 68, 44, 68, 60, 100], [28, 68, 60, 100, 0, 36], [28, 68, 60, 100, 28, 68], [28, 68, 60, 100, 60, 100], [28, 68, 60, 100, 92, 128], [28, 68, 76, 100, 92, 128], [28, 68, 92, 116, 28, 52], [28, 68, 92, 116, 28, 68], [28, 68, 92, 132, 0, 36], [28, 68, 92, 132, 28, 68], [28, 68, 92, 132, 60, 100], [28, 68, 92, 132, 92, 128], [28, 68, 124, 164, 0, 36], [28, 68, 124, 164, 28, 68], [28, 68, 124, 164, 60, 100], [28, 68, 124, 164, 92, 128], [28, 68, 140, 164, 28, 52], [28, 68, 156, 196, 0, 36], [28, 68, 156, 196, 28, 68], [28, 68, 156, 196, 60, 100], [28, 68, 156, 196, 92, 128], [28, 68, 172, 196, 92, 116], [28, 68, 188, 228, 0, 36], [28, 68, 188, 228, 28, 68], [28, 68, 188, 228, 60, 100], [28, 68, 188, 228, 92, 128], [28, 68, 220, 244, 108, 128], [28, 68, 220, 256, 0, 36], [28, 68, 220, 256, 28, 68], [28, 68, 220, 256, 60, 100], [28, 68, 220, 256, 92, 128], [28, 68, 236, 256, 76, 100], [40, 72, 0, 24, 0, 24], [40, 72, 0, 24, 8, 40], [40, 72, 0, 24, 24, 56], [40, 72, 0, 24, 40, 72], [40, 72, 0, 24, 56, 88], [40, 72, 0, 24, 72, 104], [40, 72, 0, 24, 88, 120], [40, 72, 0, 24, 104, 128], [40, 72, 0, 40, 24, 72], [40, 72, 8, 40, 0, 24], [40, 72, 8, 40, 8, 40], [40, 72, 8, 40, 24, 56], [40, 72, 8, 40, 40, 72], [40, 72, 8, 40, 56, 88], [40, 72, 8, 40, 72, 104], [40, 72, 8, 40, 88, 120], [40, 72, 8, 40, 104, 128], [40, 72, 24, 56, 0, 24], [40, 72, 24, 56, 8, 40], [40, 72, 24, 56, 24, 56], [40, 72, 24, 56, 24, 72], [40, 72, 24, 56, 40, 72], [40, 72, 24, 56, 56, 88], [40, 72, 24, 56, 72, 104], [40, 72, 24, 56, 88, 120], [40, 72, 24, 56, 104, 128], [40, 72, 40, 72, 0, 24], [40, 72, 40, 72, 8, 40], [40, 72, 40, 72, 24, 56], [40, 72, 40, 72, 40, 72], [40, 72, 40, 72, 56, 88], [40, 72, 40, 72, 72, 104], [40, 72, 40, 72, 88, 120], [40, 72, 40, 72, 104, 128], [40, 72, 56, 88, 0, 24], [40, 72, 56, 88, 8, 40], [40, 72, 56, 88, 24, 56], [40, 72, 56, 88, 40, 72], [40, 72, 56, 88, 56, 88], [40, 72, 56, 88, 72, 104], [40, 72, 56, 88, 88, 120], [40, 72, 56, 88, 104, 128], [40, 72, 72, 104, 0, 24], [40, 72, 72, 104, 8, 40], [40, 72, 72, 104, 24, 56], [40, 72, 72, 104, 24, 72], [40, 72, 72, 104, 40, 72], [40, 72, 72, 104, 56, 88], [40, 72, 72, 104, 72, 104], [40, 72, 72, 104, 88, 120], [40, 72, 72, 104, 104, 128], [40, 72, 88, 120, 0, 24], [40, 72, 88, 120, 8, 40], [40, 72, 88, 120, 24, 56], [40, 72, 88, 120, 40, 72], [40, 72, 88, 120, 56, 88], [40, 72, 88, 120, 72, 104], [40, 72, 88, 120, 88, 120], [40, 72, 88, 120, 104, 128], [40, 72, 104, 136, 0, 24], [40, 72, 104, 136, 8, 40], [40, 72, 104, 136, 24, 56], [40, 72, 104, 136, 40, 72], [40, 72, 104, 136, 56, 88], [40, 72, 104, 136, 72, 104], [40, 72, 104, 136, 88, 120], [40, 72, 104, 136, 104, 128], [40, 72, 120, 152, 0, 24], [40, 72, 120, 152, 8, 40], [40, 72, 120, 152, 24, 56], [40, 72, 120, 152, 40, 72], [40, 72, 120, 152, 56, 88], [40, 72, 120, 152, 72, 104], [40, 72, 120, 152, 88, 120], [40, 72, 120, 152, 104, 128], [40, 72, 120, 168, 56, 104], [40, 72, 136, 168, 0, 24], [40, 72, 136, 168, 8, 40], [40, 72, 136, 168, 24, 56], [40, 72, 136, 168, 40, 72], [40, 72, 136, 168, 56, 88], [40, 72, 136, 168, 56, 104], [40, 72, 136, 168, 72, 104], [40, 72, 136, 168, 88, 120], [40, 72, 136, 168, 104, 128], [40, 72, 152, 184, 0, 24], [40, 72, 152, 184, 8, 40], [40, 72, 152, 184, 24, 56], [40, 72, 152, 184, 40, 72], [40, 72, 152, 184, 56, 88], [40, 72, 152, 184, 72, 104], [40, 72, 152, 184, 88, 120], [40, 72, 152, 184, 104, 128], [40, 72, 168, 200, 0, 24], [40, 72, 168, 200, 8, 40], [40, 72, 168, 200, 24, 56], [40, 72, 168, 200, 40, 72], [40, 72, 168, 200, 56, 88], [40, 72, 168, 200, 72, 104], [40, 72, 168, 200, 88, 120], [40, 72, 168, 200, 104, 128], [40, 72, 184, 216, 0, 24], [40, 72, 184, 216, 8, 40], [40, 72, 184, 216, 24, 56], [40, 72, 184, 216, 40, 72], [40, 72, 184, 216, 56, 88], [40, 72, 184, 216, 56, 104], [40, 72, 184, 216, 72, 104], [40, 72, 184, 216, 88, 120], [40, 72, 184, 216, 88, 128], [40, 72, 184, 216, 104, 128], [40, 72, 184, 232, 0, 40], [40, 72, 200, 232, 0, 24], [40, 72, 200, 232, 8, 40], [40, 72, 200, 232, 24, 56], [40, 72, 200, 232, 40, 72], [40, 72, 200, 232, 56, 88], [40, 72, 200, 232, 72, 104], [40, 72, 200, 232, 88, 120], [40, 72, 200, 232, 104, 128], [40, 72, 216, 248, 0, 24], [40, 72, 216, 248, 0, 40], [40, 72, 216, 248, 8, 40], [40, 72, 216, 248, 24, 56], [40, 72, 216, 248, 40, 72], [40, 72, 216, 248, 56, 88], [40, 72, 216, 248, 72, 104], [40, 72, 216, 248, 88, 120], [40, 72, 216, 248, 104, 128], [40, 72, 216, 256, 0, 40], [40, 72, 232, 256, 0, 24], [40, 72, 232, 256, 8, 40], [40, 72, 232, 256, 24, 56], [40, 72, 232, 256, 40, 72], [40, 72, 232, 256, 56, 88], [40, 72, 232, 256, 72, 104], [40, 72, 232, 256, 88, 120], [40, 72, 232, 256, 104, 128], [44, 68, 0, 20, 0, 20], [44, 68, 0, 20, 12, 36], [44, 68, 0, 20, 28, 52], [44, 68, 0, 20, 44, 68], [44, 68, 0, 20, 60, 84], [44, 68, 0, 20, 60, 100], [44, 68, 0, 20, 76, 100], [44, 68, 0, 20, 92, 116], [44, 68, 0, 20, 108, 128], [44, 68, 12, 36, 0, 20], [44, 68, 12, 36, 12, 36], [44, 68, 12, 36, 28, 52], [44, 68, 12, 36, 44, 68], [44, 68, 12, 36, 60, 84], [44, 68, 12, 36, 76, 100], [44, 68, 12, 36, 92, 116], [44, 68, 12, 36, 108, 128], [44, 68, 28, 52, 0, 20], [44, 68, 28, 52, 12, 36], [44, 68, 28, 52, 28, 52], [44, 68, 28, 52, 44, 68], [44, 68, 28, 52, 60, 84], [44, 68, 28, 52, 76, 100], [44, 68, 28, 52, 92, 116], [44, 68, 28, 52, 108, 128], [44, 68, 44, 68, 0, 20], [44, 68, 44, 68, 12, 36], [44, 68, 44, 68, 28, 52], [44, 68, 44, 68, 44, 68], [44, 68, 44, 68, 60, 84], [44, 68, 44, 68, 60, 100], [44, 68, 44, 68, 76, 100], [44, 68, 44, 68, 92, 116], [44, 68, 44, 68, 108, 128], [44, 68, 60, 84, 0, 20], [44, 68, 60, 84, 12, 36], [44, 68, 60, 84, 28, 52], [44, 68, 60, 84, 44, 68], [44, 68, 60, 84, 60, 84], [44, 68, 60, 84, 76, 100], [44, 68, 60, 84, 92, 116], [44, 68, 60, 84, 108, 128], [44, 68, 76, 100, 0, 20], [44, 68, 76, 100, 12, 36], [44, 68, 76, 100, 28, 52], [44, 68, 76, 100, 44, 68], [44, 68, 76, 100, 60, 84], [44, 68, 76, 100, 76, 100], [44, 68, 76, 100, 92, 116], [44, 68, 76, 100, 108, 128], [44, 68, 92, 116, 0, 20], [44, 68, 92, 116, 12, 36], [44, 68, 92, 116, 28, 52], [44, 68, 92, 116, 44, 68], [44, 68, 92, 116, 60, 84], [44, 68, 92, 116, 76, 100], [44, 68, 92, 116, 92, 116], [44, 68, 92, 116, 108, 128], [44, 68, 108, 132, 0, 20], [44, 68, 108, 132, 12, 36], [44, 68, 108, 132, 28, 52], [44, 68, 108, 132, 44, 68], [44, 68, 108, 132, 60, 84], [44, 68, 108, 132, 76, 100], [44, 68, 108, 132, 92, 116], [44, 68, 108, 132, 108, 128], [44, 68, 124, 148, 0, 20], [44, 68, 124, 148, 12, 36], [44, 68, 124, 148, 28, 52], [44, 68, 124, 148, 44, 68], [44, 68, 124, 148, 60, 84], [44, 68, 124, 148, 76, 100], [44, 68, 124, 148, 92, 116], [44, 68, 124, 148, 108, 128], [44, 68, 140, 164, 0, 20], [44, 68, 140, 164, 12, 36], [44, 68, 140, 164, 28, 52], [44, 68, 140, 164, 44, 68], [44, 68, 140, 164, 60, 84], [44, 68, 140, 164, 76, 100], [44, 68, 140, 164, 92, 116], [44, 68, 140, 164, 108, 128], [44, 68, 156, 180, 0, 20], [44, 68, 156, 180, 12, 36], [44, 68, 156, 180, 28, 52], [44, 68, 156, 180, 44, 68], [44, 68, 156, 180, 60, 84], [44, 68, 156, 180, 76, 100], [44, 68, 156, 180, 92, 116], [44, 68, 156, 180, 108, 128], [44, 68, 172, 196, 0, 20], [44, 68, 172, 196, 12, 36], [44, 68, 172, 196, 28, 52], [44, 68, 172, 196, 44, 68], [44, 68, 172, 196, 60, 84], [44, 68, 172, 196, 76, 100], [44, 68, 172, 196, 92, 116], [44, 68, 172, 196, 108, 128], [44, 68, 188, 212, 0, 20], [44, 68, 188, 212, 12, 36], [44, 68, 188, 212, 28, 52], [44, 68, 188, 212, 44, 68], [44, 68, 188, 212, 60, 84], [44, 68, 188, 212, 76, 100], [44, 68, 188, 212, 92, 116], [44, 68, 188, 212, 108, 128], [44, 68, 204, 228, 0, 20], [44, 68, 204, 228, 12, 36], [44, 68, 204, 228, 28, 52], [44, 68, 204, 228, 44, 68], [44, 68, 204, 228, 60, 84], [44, 68, 204, 228, 76, 100], [44, 68, 204, 228, 92, 116], [44, 68, 204, 228, 108, 128], [44, 68, 220, 244, 0, 20], [44, 68, 220, 244, 12, 36], [44, 68, 220, 244, 28, 52], [44, 68, 220, 244, 44, 68], [44, 68, 220, 244, 60, 84], [44, 68, 220, 244, 76, 100], [44, 68, 220, 244, 92, 116], [44, 68, 220, 244, 108, 128], [44, 68, 220, 256, 0, 36], [44, 68, 236, 256, 0, 20], [44, 68, 236, 256, 12, 36], [44, 68, 236, 256, 28, 52], [44, 68, 236, 256, 44, 68], [44, 68, 236, 256, 60, 84], [44, 68, 236, 256, 76, 100], [44, 68, 236, 256, 92, 116], [44, 68, 236, 256, 108, 128], [56, 88, 0, 24, 0, 24], [56, 88, 0, 24, 8, 40], [56, 88, 0, 24, 24, 56], [56, 88, 0, 24, 40, 72], [56, 88, 0, 24, 56, 88], [56, 88, 0, 24, 72, 104], [56, 88, 0, 24, 88, 120], [56, 88, 0, 24, 104, 128], [56, 88, 8, 40, 0, 24], [56, 88, 8, 40, 8, 40], [56, 88, 8, 40, 24, 56], [56, 88, 8, 40, 40, 72], [56, 88, 8, 40, 56, 88], [56, 88, 8, 40, 72, 104], [56, 88, 8, 40, 88, 120], [56, 88, 8, 40, 104, 128], [56, 88, 24, 56, 0, 24], [56, 88, 24, 56, 8, 40], [56, 88, 24, 56, 24, 56], [56, 88, 24, 56, 40, 72], [56, 88, 24, 56, 56, 88], [56, 88, 24, 56, 72, 104], [56, 88, 24, 56, 88, 120], [56, 88, 24, 56, 104, 128], [56, 88, 40, 72, 0, 24], [56, 88, 40, 72, 8, 40], [56, 88, 40, 72, 24, 56], [56, 88, 40, 72, 40, 72], [56, 88, 40, 72, 56, 88], [56, 88, 40, 72, 72, 104], [56, 88, 40, 72, 88, 120], [56, 88, 40, 72, 104, 128], [56, 88, 56, 88, 0, 24], [56, 88, 56, 88, 8, 40], [56, 88, 56, 88, 24, 56], [56, 88, 56, 88, 40, 72], [56, 88, 56, 88, 56, 88], [56, 88, 56, 88, 72, 104], [56, 88, 56, 88, 88, 120], [56, 88, 56, 88, 104, 128], [56, 88, 72, 104, 0, 24], [56, 88, 72, 104, 8, 40], [56, 88, 72, 104, 24, 56], [56, 88, 72, 104, 40, 72], [56, 88, 72, 104, 56, 88], [56, 88, 72, 104, 72, 104], [56, 88, 72, 104, 88, 120], [56, 88, 72, 104, 104, 128], [56, 88, 88, 120, 0, 24], [56, 88, 88, 120, 8, 40], [56, 88, 88, 120, 24, 56], [56, 88, 88, 120, 40, 72], [56, 88, 88, 120, 56, 88], [56, 88, 88, 120, 72, 104], [56, 88, 88, 120, 88, 120], [56, 88, 88, 120, 104, 128], [56, 88, 104, 136, 0, 24], [56, 88, 104, 136, 8, 40], [56, 88, 104, 136, 24, 56], [56, 88, 104, 136, 40, 72], [56, 88, 104, 136, 56, 88], [56, 88, 104, 136, 72, 104], [56, 88, 104, 136, 88, 120], [56, 88, 104, 136, 104, 128], [56, 88, 120, 152, 0, 24], [56, 88, 120, 152, 8, 40], [56, 88, 120, 152, 24, 56], [56, 88, 120, 152, 40, 72], [56, 88, 120, 152, 56, 88], [56, 88, 120, 152, 72, 104], [56, 88, 120, 152, 88, 120], [56, 88, 120, 152, 104, 128], [56, 88, 136, 168, 0, 24], [56, 88, 136, 168, 8, 40], [56, 88, 136, 168, 24, 56], [56, 88, 136, 168, 40, 72], [56, 88, 136, 168, 56, 88], [56, 88, 136, 168, 56, 104], [56, 88, 136, 168, 72, 104], [56, 88, 136, 168, 88, 120], [56, 88, 136, 168, 104, 128], [56, 88, 152, 184, 0, 24], [56, 88, 152, 184, 8, 40], [56, 88, 152, 184, 24, 56], [56, 88, 152, 184, 40, 72], [56, 88, 152, 184, 56, 88], [56, 88, 152, 184, 72, 104], [56, 88, 152, 184, 88, 120], [56, 88, 152, 184, 104, 128], [56, 88, 152, 200, 24, 72], [56, 88, 168, 200, 0, 24], [56, 88, 168, 200, 8, 40], [56, 88, 168, 200, 24, 56], [56, 88, 168, 200, 40, 72], [56, 88, 168, 200, 56, 88], [56, 88, 168, 200, 72, 104], [56, 88, 168, 200, 88, 120], [56, 88, 168, 200, 104, 128], [56, 88, 184, 216, 0, 24], [56, 88, 184, 216, 8, 40], [56, 88, 184, 216, 24, 56], [56, 88, 184, 216, 40, 72], [56, 88, 184, 216, 56, 88], [56, 88, 184, 216, 72, 104], [56, 88, 184, 216, 88, 120], [56, 88, 184, 216, 104, 128], [56, 88, 200, 232, 0, 24], [56, 88, 200, 232, 8, 40], [56, 88, 200, 232, 24, 56], [56, 88, 200, 232, 40, 72], [56, 88, 200, 232, 56, 88], [56, 88, 200, 232, 72, 104], [56, 88, 200, 232, 88, 120], [56, 88, 200, 232, 104, 128], [56, 88, 216, 248, 0, 24], [56, 88, 216, 248, 8, 40], [56, 88, 216, 248, 24, 56], [56, 88, 216, 248, 40, 72], [56, 88, 216, 248, 56, 88], [56, 88, 216, 248, 72, 104], [56, 88, 216, 248, 88, 120], [56, 88, 216, 248, 104, 128], [56, 88, 232, 256, 0, 24], [56, 88, 232, 256, 8, 40], [56, 88, 232, 256, 24, 56], [56, 88, 232, 256, 40, 72], [56, 88, 232, 256, 56, 88], [56, 88, 232, 256, 72, 104], [56, 88, 232, 256, 88, 120], [56, 88, 232, 256, 104, 128], [56, 104, 0, 40, 0, 40], [56, 104, 0, 40, 24, 72], [56, 104, 0, 40, 56, 88], [56, 104, 0, 40, 56, 104], [56, 104, 0, 40, 88, 128], [56, 104, 8, 40, 88, 128], [56, 104, 24, 72, 0, 40], [56, 104, 24, 72, 24, 72], [56, 104, 24, 72, 56, 104], [56, 104, 24, 72, 88, 128], [56, 104, 56, 88, 0, 40], [56, 104, 56, 104, 0, 40], [56, 104, 56, 104, 24, 72], [56, 104, 56, 104, 56, 104], [56, 104, 56, 104, 88, 128], [56, 104, 72, 104, 40, 72], [56, 104, 72, 104, 88, 128], [56, 104, 88, 120, 0, 24], [56, 104, 88, 120, 56, 88], [56, 104, 88, 120, 56, 104], [56, 104, 88, 120, 88, 128], [56, 104, 88, 136, 0, 40], [56, 104, 88, 136, 24, 72], [56, 104, 88, 136, 40, 72], [56, 104, 88, 136, 56, 104], [56, 104, 88, 136, 88, 128], [56, 104, 120, 168, 0, 40], [56, 104, 120, 168, 24, 72], [56, 104, 120, 168, 40, 72], [56, 104, 120, 168, 56, 104], [56, 104, 120, 168, 88, 128], [56, 104, 152, 200, 0, 40], [56, 104, 152, 200, 24, 72], [56, 104, 152, 200, 56, 104], [56, 104, 152, 200, 88, 128], [56, 104, 168, 200, 0, 40], [56, 104, 184, 232, 0, 40], [56, 104, 184, 232, 24, 72], [56, 104, 184, 232, 56, 104], [56, 104, 184, 232, 88, 128], [56, 104, 216, 248, 104, 128], [56, 104, 216, 256, 0, 40], [56, 104, 216, 256, 24, 72], [56, 104, 216, 256, 56, 104], [56, 104, 216, 256, 88, 128], [56, 104, 232, 256, 0, 24], [56, 104, 232, 256, 0, 40], [56, 125, 0, 72, 0, 72], [56, 125, 0, 72, 56, 128], [56, 125, 56, 104, 56, 128], [56, 125, 56, 120, 56, 128], [56, 125, 56, 136, 0, 72], [56, 125, 56, 136, 56, 128], [56, 125, 120, 200, 0, 72], [56, 125, 120, 200, 56, 128], [56, 125, 136, 200, 56, 128], [56, 125, 184, 256, 0, 72], [56, 125, 184, 256, 56, 128], [60, 84, 0, 20, 0, 20], [60, 84, 0, 20, 12, 36], [60, 84, 0, 20, 28, 52], [60, 84, 0, 20, 44, 68], [60, 84, 0, 20, 60, 84], [60, 84, 0, 20, 76, 100], [60, 84, 0, 20, 92, 116], [60, 84, 0, 20, 108, 128], [60, 84, 12, 36, 0, 20], [60, 84, 12, 36, 12, 36], [60, 84, 12, 36, 28, 52], [60, 84, 12, 36, 44, 68], [60, 84, 12, 36, 60, 84], [60, 84, 12, 36, 76, 100], [60, 84, 12, 36, 92, 116], [60, 84, 12, 36, 108, 128], [60, 84, 28, 52, 0, 20], [60, 84, 28, 52, 12, 36], [60, 84, 28, 52, 28, 52], [60, 84, 28, 52, 44, 68], [60, 84, 28, 52, 60, 84], [60, 84, 28, 52, 76, 100], [60, 84, 28, 52, 92, 116], [60, 84, 28, 52, 108, 128], [60, 84, 44, 68, 0, 20], [60, 84, 44, 68, 12, 36], [60, 84, 44, 68, 28, 52], [60, 84, 44, 68, 44, 68], [60, 84, 44, 68, 60, 84], [60, 84, 44, 68, 76, 100], [60, 84, 44, 68, 92, 116], [60, 84, 44, 68, 108, 128], [60, 84, 60, 84, 0, 20], [60, 84, 60, 84, 12, 36], [60, 84, 60, 84, 28, 52], [60, 84, 60, 84, 44, 68], [60, 84, 60, 84, 60, 84], [60, 84, 60, 84, 76, 100], [60, 84, 60, 84, 92, 116], [60, 84, 60, 84, 108, 128], [60, 84, 76, 100, 0, 20], [60, 84, 76, 100, 12, 36], [60, 84, 76, 100, 28, 52], [60, 84, 76, 100, 44, 68], [60, 84, 76, 100, 60, 84], [60, 84, 76, 100, 76, 100], [60, 84, 76, 100, 92, 116], [60, 84, 76, 100, 108, 128], [60, 84, 92, 116, 0, 20], [60, 84, 92, 116, 12, 36], [60, 84, 92, 116, 28, 52], [60, 84, 92, 116, 44, 68], [60, 84, 92, 116, 60, 84], [60, 84, 92, 116, 76, 100], [60, 84, 92, 116, 92, 116], [60, 84, 92, 116, 108, 128], [60, 84, 108, 132, 0, 20], [60, 84, 108, 132, 12, 36], [60, 84, 108, 132, 28, 52], [60, 84, 108, 132, 44, 68], [60, 84, 108, 132, 60, 84], [60, 84, 108, 132, 76, 100], [60, 84, 108, 132, 92, 116], [60, 84, 108, 132, 108, 128], [60, 84, 124, 148, 0, 20], [60, 84, 124, 148, 12, 36], [60, 84, 124, 148, 28, 52], [60, 84, 124, 148, 44, 68], [60, 84, 124, 148, 60, 84], [60, 84, 124, 148, 76, 100], [60, 84, 124, 148, 92, 116], [60, 84, 124, 148, 108, 128], [60, 84, 140, 164, 0, 20], [60, 84, 140, 164, 12, 36], [60, 84, 140, 164, 28, 52], [60, 84, 140, 164, 44, 68], [60, 84, 140, 164, 60, 84], [60, 84, 140, 164, 60, 100], [60, 84, 140, 164, 76, 100], [60, 84, 140, 164, 92, 116], [60, 84, 140, 164, 108, 128], [60, 84, 156, 180, 0, 20], [60, 84, 156, 180, 12, 36], [60, 84, 156, 180, 28, 52], [60, 84, 156, 180, 44, 68], [60, 84, 156, 180, 60, 84], [60, 84, 156, 180, 76, 100], [60, 84, 156, 180, 92, 116], [60, 84, 156, 180, 108, 128], [60, 84, 172, 196, 0, 20], [60, 84, 172, 196, 12, 36], [60, 84, 172, 196, 28, 52], [60, 84, 172, 196, 44, 68], [60, 84, 172, 196, 60, 84], [60, 84, 172, 196, 76, 100], [60, 84, 172, 196, 92, 116], [60, 84, 172, 196, 108, 128], [60, 84, 188, 212, 0, 20], [60, 84, 188, 212, 12, 36], [60, 84, 188, 212, 28, 52], [60, 84, 188, 212, 44, 68], [60, 84, 188, 212, 60, 84], [60, 84, 188, 212, 76, 100], [60, 84, 188, 212, 92, 116], [60, 84, 188, 212, 108, 128], [60, 84, 204, 228, 0, 20], [60, 84, 204, 228, 12, 36], [60, 84, 204, 228, 28, 52], [60, 84, 204, 228, 44, 68], [60, 84, 204, 228, 60, 84], [60, 84, 204, 228, 76, 100], [60, 84, 204, 228, 92, 116], [60, 84, 204, 228, 108, 128], [60, 84, 220, 244, 0, 20], [60, 84, 220, 244, 12, 36], [60, 84, 220, 244, 28, 52], [60, 84, 220, 244, 44, 68], [60, 84, 220, 244, 60, 84], [60, 84, 220, 244, 76, 100], [60, 84, 220, 244, 92, 116], [60, 84, 220, 244, 108, 128], [60, 84, 236, 256, 0, 20], [60, 84, 236, 256, 12, 36], [60, 84, 236, 256, 28, 52], [60, 84, 236, 256, 44, 68], [60, 84, 236, 256, 60, 84], [60, 84, 236, 256, 76, 100], [60, 84, 236, 256, 92, 116], [60, 84, 236, 256, 108, 128], [60, 100, 0, 36, 0, 36], [60, 100, 0, 36, 28, 68], [60, 100, 0, 36, 60, 100], [60, 100, 0, 36, 92, 128], [60, 100, 12, 36, 60, 84], [60, 100, 12, 36, 92, 128], [60, 100, 28, 68, 0, 36], [60, 100, 28, 68, 12, 36], [60, 100, 28, 68, 28, 68], [60, 100, 28, 68, 60, 84], [60, 100, 28, 68, 60, 100], [60, 100, 28, 68, 92, 128], [60, 100, 44, 68, 28, 68], [60, 100, 60, 84, 0, 36], [60, 100, 60, 84, 12, 36], [60, 100, 60, 100, 0, 36], [60, 100, 60, 100, 28, 68], [60, 100, 60, 100, 60, 100], [60, 100, 60, 100, 92, 128], [60, 100, 92, 116, 60, 84], [60, 100, 92, 132, 0, 36], [60, 100, 92, 132, 28, 68], [60, 100, 92, 132, 60, 100], [60, 100, 92, 132, 92, 128], [60, 100, 124, 164, 0, 36], [60, 100, 124, 164, 12, 36], [60, 100, 124, 164, 28, 68], [60, 100, 124, 164, 60, 100], [60, 100, 124, 164, 92, 128], [60, 100, 140, 164, 12, 36], [60, 100, 140, 164, 60, 100], [60, 100, 156, 196, 0, 36], [60, 100, 156, 196, 28, 68], [60, 100, 156, 196, 60, 100], [60, 100, 156, 196, 92, 116], [60, 100, 156, 196, 92, 128], [60, 100, 172, 196, 92, 116], [60, 100, 188, 228, 0, 36], [60, 100, 188, 228, 28, 52], [60, 100, 188, 228, 28, 68], [60, 100, 188, 228, 60, 100], [60, 100, 188, 228, 92, 128], [60, 100, 204, 228, 92, 116], [60, 100, 220, 244, 28, 52], [60, 100, 220, 244, 28, 68], [60, 100, 220, 244, 60, 100], [60, 100, 220, 256, 0, 36], [60, 100, 220, 256, 28, 68], [60, 100, 220, 256, 60, 100], [60, 100, 220, 256, 92, 128], [60, 100, 236, 256, 0, 36], [60, 125, 0, 36, 0, 68], [60, 125, 0, 68, 0, 68], [60, 125, 0, 68, 60, 128], [60, 125, 60, 100, 60, 128], [60, 125, 60, 116, 60, 128], [60, 125, 60, 132, 0, 68], [60, 125, 60, 132, 12, 68], [60, 125, 60, 132, 60, 128], [60, 125, 124, 196, 0, 68], [60, 125, 124, 196, 60, 128], [60, 125, 188, 256, 0, 68], [60, 125, 188, 256, 60, 128], [60, 125, 204, 256, 60, 128], [72, 104, 0, 24, 0, 24], [72, 104, 0, 24, 8, 40], [72, 104, 0, 24, 24, 56], [72, 104, 0, 24, 40, 72], [72, 104, 0, 24, 56, 88], [72, 104, 0, 24, 72, 104], [72, 104, 0, 24, 88, 120], [72, 104, 0, 24, 104, 128], [72, 104, 8, 40, 0, 24], [72, 104, 8, 40, 8, 40], [72, 104, 8, 40, 24, 56], [72, 104, 8, 40, 40, 72], [72, 104, 8, 40, 56, 88], [72, 104, 8, 40, 72, 104], [72, 104, 8, 40, 88, 120], [72, 104, 8, 40, 88, 128], [72, 104, 8, 40, 104, 128], [72, 104, 24, 56, 0, 24], [72, 104, 24, 56, 8, 40], [72, 104, 24, 56, 24, 56], [72, 104, 24, 56, 40, 72], [72, 104, 24, 56, 56, 88], [72, 104, 24, 56, 72, 104], [72, 104, 24, 56, 88, 120], [72, 104, 24, 56, 104, 128], [72, 104, 24, 72, 0, 40], [72, 104, 24, 72, 56, 104], [72, 104, 24, 72, 88, 128], [72, 104, 40, 72, 0, 24], [72, 104, 40, 72, 8, 40], [72, 104, 40, 72, 24, 56], [72, 104, 40, 72, 40, 72], [72, 104, 40, 72, 56, 88], [72, 104, 40, 72, 72, 104], [72, 104, 40, 72, 88, 120], [72, 104, 40, 72, 104, 128], [72, 104, 56, 88, 0, 24], [72, 104, 56, 88, 0, 40], [72, 104, 56, 88, 8, 40], [72, 104, 56, 88, 24, 56], [72, 104, 56, 88, 40, 72], [72, 104, 56, 88, 56, 88], [72, 104, 56, 88, 56, 104], [72, 104, 56, 88, 72, 104], [72, 104, 56, 88, 88, 120], [72, 104, 56, 88, 104, 128], [72, 104, 56, 104, 56, 104], [72, 104, 72, 104, 0, 24], [72, 104, 72, 104, 8, 40], [72, 104, 72, 104, 24, 56], [72, 104, 72, 104, 40, 72], [72, 104, 72, 104, 56, 88], [72, 104, 72, 104, 72, 104], [72, 104, 72, 104, 88, 120], [72, 104, 72, 104, 104, 128], [72, 104, 88, 120, 0, 24], [72, 104, 88, 120, 8, 40], [72, 104, 88, 120, 24, 56], [72, 104, 88, 120, 40, 72], [72, 104, 88, 120, 56, 88], [72, 104, 88, 120, 72, 104], [72, 104, 88, 120, 88, 120], [72, 104, 88, 120, 104, 128], [72, 104, 104, 136, 0, 24], [72, 104, 104, 136, 8, 40], [72, 104, 104, 136, 24, 56], [72, 104, 104, 136, 40, 72], [72, 104, 104, 136, 56, 88], [72, 104, 104, 136, 72, 104], [72, 104, 104, 136, 88, 120], [72, 104, 104, 136, 104, 128], [72, 104, 120, 152, 0, 24], [72, 104, 120, 152, 8, 40], [72, 104, 120, 152, 24, 56], [72, 104, 120, 152, 40, 72], [72, 104, 120, 152, 56, 88], [72, 104, 120, 152, 72, 104], [72, 104, 120, 152, 88, 120], [72, 104, 120, 152, 104, 128], [72, 104, 120, 168, 0, 40], [72, 104, 136, 168, 0, 24], [72, 104, 136, 168, 8, 40], [72, 104, 136, 168, 24, 56], [72, 104, 136, 168, 40, 72], [72, 104, 136, 168, 56, 88], [72, 104, 136, 168, 72, 104], [72, 104, 136, 168, 88, 120], [72, 104, 136, 168, 104, 128], [72, 104, 152, 184, 0, 24], [72, 104, 152, 184, 8, 40], [72, 104, 152, 184, 24, 56], [72, 104, 152, 184, 40, 72], [72, 104, 152, 184, 56, 88], [72, 104, 152, 184, 72, 104], [72, 104, 152, 184, 88, 120], [72, 104, 152, 184, 104, 128], [72, 104, 168, 200, 0, 24], [72, 104, 168, 200, 8, 40], [72, 104, 168, 200, 24, 56], [72, 104, 168, 200, 40, 72], [72, 104, 168, 200, 56, 88], [72, 104, 168, 200, 72, 104], [72, 104, 168, 200, 88, 120], [72, 104, 168, 200, 104, 128], [72, 104, 184, 216, 0, 24], [72, 104, 184, 216, 8, 40], [72, 104, 184, 216, 24, 56], [72, 104, 184, 216, 40, 72], [72, 104, 184, 216, 56, 88], [72, 104, 184, 216, 72, 104], [72, 104, 184, 216, 88, 120], [72, 104, 184, 216, 104, 128], [72, 104, 184, 232, 24, 56], [72, 104, 200, 232, 0, 24], [72, 104, 200, 232, 8, 40], [72, 104, 200, 232, 24, 56], [72, 104, 200, 232, 40, 72], [72, 104, 200, 232, 56, 88], [72, 104, 200, 232, 72, 104], [72, 104, 200, 232, 88, 120], [72, 104, 200, 232, 104, 128], [72, 104, 216, 248, 0, 24], [72, 104, 216, 248, 8, 40], [72, 104, 216, 248, 24, 56], [72, 104, 216, 248, 40, 72], [72, 104, 216, 248, 56, 88], [72, 104, 216, 248, 72, 104], [72, 104, 216, 248, 88, 120], [72, 104, 216, 248, 104, 128], [72, 104, 232, 256, 0, 24], [72, 104, 232, 256, 8, 40], [72, 104, 232, 256, 24, 56], [72, 104, 232, 256, 40, 72], [72, 104, 232, 256, 56, 88], [72, 104, 232, 256, 72, 104], [72, 104, 232, 256, 88, 120], [72, 104, 232, 256, 104, 128], [72, 125, 0, 72, 56, 128], [76, 100, 0, 20, 0, 20], [76, 100, 0, 20, 12, 36], [76, 100, 0, 20, 28, 52], [76, 100, 0, 20, 44, 68], [76, 100, 0, 20, 60, 84], [76, 100, 0, 20, 76, 100], [76, 100, 0, 20, 92, 116], [76, 100, 0, 20, 108, 128], [76, 100, 0, 36, 0, 36], [76, 100, 12, 36, 0, 20], [76, 100, 12, 36, 12, 36], [76, 100, 12, 36, 28, 52], [76, 100, 12, 36, 28, 68], [76, 100, 12, 36, 44, 68], [76, 100, 12, 36, 60, 84], [76, 100, 12, 36, 76, 100], [76, 100, 12, 36, 92, 116], [76, 100, 12, 36, 92, 128], [76, 100, 12, 36, 108, 128], [76, 100, 28, 52, 0, 20], [76, 100, 28, 52, 12, 36], [76, 100, 28, 52, 28, 52], [76, 100, 28, 52, 44, 68], [76, 100, 28, 52, 60, 84], [76, 100, 28, 52, 76, 100], [76, 100, 28, 52, 92, 116], [76, 100, 28, 52, 108, 128], [76, 100, 44, 68, 0, 20], [76, 100, 44, 68, 12, 36], [76, 100, 44, 68, 28, 52], [76, 100, 44, 68, 44, 68], [76, 100, 44, 68, 60, 84], [76, 100, 44, 68, 76, 100], [76, 100, 44, 68, 92, 116], [76, 100, 44, 68, 108, 128], [76, 100, 60, 84, 0, 20], [76, 100, 60, 84, 12, 36], [76, 100, 60, 84, 28, 52], [76, 100, 60, 84, 44, 68], [76, 100, 60, 84, 60, 84], [76, 100, 60, 84, 76, 100], [76, 100, 60, 84, 92, 116], [76, 100, 60, 84, 108, 128], [76, 100, 76, 100, 0, 20], [76, 100, 76, 100, 12, 36], [76, 100, 76, 100, 28, 52], [76, 100, 76, 100, 44, 68], [76, 100, 76, 100, 60, 84], [76, 100, 76, 100, 76, 100], [76, 100, 76, 100, 92, 116], [76, 100, 76, 100, 108, 128], [76, 100, 92, 116, 0, 20], [76, 100, 92, 116, 12, 36], [76, 100, 92, 116, 28, 52], [76, 100, 92, 116, 44, 68], [76, 100, 92, 116, 60, 84], [76, 100, 92, 116, 76, 100], [76, 100, 92, 116, 92, 116], [76, 100, 92, 116, 108, 128], [76, 100, 92, 132, 92, 128], [76, 100, 92, 132, 108, 128], [76, 100, 108, 132, 0, 20], [76, 100, 108, 132, 0, 36], [76, 100, 108, 132, 12, 36], [76, 100, 108, 132, 28, 52], [76, 100, 108, 132, 28, 68], [76, 100, 108, 132, 44, 68], [76, 100, 108, 132, 60, 84], [76, 100, 108, 132, 76, 100], [76, 100, 108, 132, 92, 116], [76, 100, 108, 132, 108, 128], [76, 100, 124, 148, 0, 20], [76, 100, 124, 148, 12, 36], [76, 100, 124, 148, 28, 52], [76, 100, 124, 148, 44, 68], [76, 100, 124, 148, 60, 84], [76, 100, 124, 148, 76, 100], [76, 100, 124, 148, 92, 116], [76, 100, 124, 148, 108, 128], [76, 100, 140, 164, 0, 20], [76, 100, 140, 164, 12, 36], [76, 100, 140, 164, 28, 52], [76, 100, 140, 164, 44, 68], [76, 100, 140, 164, 60, 84], [76, 100, 140, 164, 76, 100], [76, 100, 140, 164, 92, 116], [76, 100, 140, 164, 108, 128], [76, 100, 156, 180, 0, 20], [76, 100, 156, 180, 12, 36], [76, 100, 156, 180, 28, 52], [76, 100, 156, 180, 44, 68], [76, 100, 156, 180, 60, 84], [76, 100, 156, 180, 76, 100], [76, 100, 156, 180, 92, 116], [76, 100, 156, 180, 108, 128], [76, 100, 156, 196, 0, 36], [76, 100, 172, 196, 0, 20], [76, 100, 172, 196, 0, 36], [76, 100, 172, 196, 12, 36], [76, 100, 172, 196, 28, 52], [76, 100, 172, 196, 44, 68], [76, 100, 172, 196, 60, 84], [76, 100, 172, 196, 76, 100], [76, 100, 172, 196, 92, 116], [76, 100, 172, 196, 108, 128], [76, 100, 188, 212, 0, 20], [76, 100, 188, 212, 12, 36], [76, 100, 188, 212, 28, 52], [76, 100, 188, 212, 44, 68], [76, 100, 188, 212, 60, 84], [76, 100, 188, 212, 76, 100], [76, 100, 188, 212, 92, 116], [76, 100, 188, 212, 108, 128], [76, 100, 204, 228, 0, 20], [76, 100, 204, 228, 12, 36], [76, 100, 204, 228, 28, 52], [76, 100, 204, 228, 44, 68], [76, 100, 204, 228, 60, 84], [76, 100, 204, 228, 76, 100], [76, 100, 204, 228, 92, 116], [76, 100, 204, 228, 108, 128], [76, 100, 220, 244, 0, 20], [76, 100, 220, 244, 12, 36], [76, 100, 220, 244, 28, 52], [76, 100, 220, 244, 44, 68], [76, 100, 220, 244, 60, 84], [76, 100, 220, 244, 76, 100], [76, 100, 220, 244, 92, 116], [76, 100, 220, 244, 108, 128], [76, 100, 236, 256, 0, 20], [76, 100, 236, 256, 12, 36], [76, 100, 236, 256, 28, 52], [76, 100, 236, 256, 44, 68], [76, 100, 236, 256, 60, 84], [76, 100, 236, 256, 76, 100], [76, 100, 236, 256, 92, 116], [76, 100, 236, 256, 108, 128], [76, 125, 0, 68, 0, 68], [76, 125, 60, 116, 12, 68], [76, 125, 124, 196, 0, 68], [88, 120, 0, 24, 0, 24], [88, 120, 0, 24, 8, 40], [88, 120, 0, 24, 24, 56], [88, 120, 0, 24, 40, 72], [88, 120, 0, 24, 56, 88], [88, 120, 0, 24, 72, 104], [88, 120, 0, 24, 88, 120], [88, 120, 0, 24, 104, 128], [88, 120, 0, 40, 24, 72], [88, 120, 8, 40, 0, 24], [88, 120, 8, 40, 8, 40], [88, 120, 8, 40, 24, 56], [88, 120, 8, 40, 40, 72], [88, 120, 8, 40, 56, 88], [88, 120, 8, 40, 72, 104], [88, 120, 8, 40, 88, 120], [88, 120, 8, 40, 88, 128], [88, 120, 8, 40, 104, 128], [88, 120, 24, 56, 0, 24], [88, 120, 24, 56, 0, 40], [88, 120, 24, 56, 8, 40], [88, 120, 24, 56, 24, 56], [88, 120, 24, 56, 40, 72], [88, 120, 24, 56, 56, 88], [88, 120, 24, 56, 72, 104], [88, 120, 24, 56, 88, 120], [88, 120, 24, 56, 104, 128], [88, 120, 24, 72, 56, 88], [88, 120, 40, 72, 0, 24], [88, 120, 40, 72, 8, 40], [88, 120, 40, 72, 24, 56], [88, 120, 40, 72, 40, 72], [88, 120, 40, 72, 56, 88], [88, 120, 40, 72, 72, 104], [88, 120, 40, 72, 88, 120], [88, 120, 40, 72, 104, 128], [88, 120, 56, 88, 0, 24], [88, 120, 56, 88, 8, 40], [88, 120, 56, 88, 24, 56], [88, 120, 56, 88, 40, 72], [88, 120, 56, 88, 56, 88], [88, 120, 56, 88, 72, 104], [88, 120, 56, 88, 88, 120], [88, 120, 56, 88, 104, 128], [88, 120, 72, 104, 0, 24], [88, 120, 72, 104, 8, 40], [88, 120, 72, 104, 24, 56], [88, 120, 72, 104, 40, 72], [88, 120, 72, 104, 56, 88], [88, 120, 72, 104, 72, 104], [88, 120, 72, 104, 88, 120], [88, 120, 72, 104, 104, 128], [88, 120, 88, 120, 0, 24], [88, 120, 88, 120, 8, 40], [88, 120, 88, 120, 24, 56], [88, 120, 88, 120, 40, 72], [88, 120, 88, 120, 56, 88], [88, 120, 88, 120, 72, 104], [88, 120, 88, 120, 88, 120], [88, 120, 88, 120, 104, 128], [88, 120, 88, 136, 88, 128], [88, 120, 104, 136, 0, 24], [88, 120, 104, 136, 8, 40], [88, 120, 104, 136, 24, 56], [88, 120, 104, 136, 40, 72], [88, 120, 104, 136, 56, 88], [88, 120, 104, 136, 56, 104], [88, 120, 104, 136, 72, 104], [88, 120, 104, 136, 88, 120], [88, 120, 104, 136, 104, 128], [88, 120, 120, 152, 0, 24], [88, 120, 120, 152, 8, 40], [88, 120, 120, 152, 24, 56], [88, 120, 120, 152, 40, 72], [88, 120, 120, 152, 56, 88], [88, 120, 120, 152, 72, 104], [88, 120, 120, 152, 88, 120], [88, 120, 120, 152, 104, 128], [88, 120, 136, 168, 0, 24], [88, 120, 136, 168, 8, 40], [88, 120, 136, 168, 24, 56], [88, 120, 136, 168, 40, 72], [88, 120, 136, 168, 56, 88], [88, 120, 136, 168, 72, 104], [88, 120, 136, 168, 88, 120], [88, 120, 136, 168, 104, 128], [88, 120, 152, 184, 0, 24], [88, 120, 152, 184, 8, 40], [88, 120, 152, 184, 24, 56], [88, 120, 152, 184, 40, 72], [88, 120, 152, 184, 56, 88], [88, 120, 152, 184, 72, 104], [88, 120, 152, 184, 88, 120], [88, 120, 152, 184, 104, 128], [88, 120, 168, 200, 0, 24], [88, 120, 168, 200, 8, 40], [88, 120, 168, 200, 24, 56], [88, 120, 168, 200, 40, 72], [88, 120, 168, 200, 56, 88], [88, 120, 168, 200, 72, 104], [88, 120, 168, 200, 88, 120], [88, 120, 168, 200, 104, 128], [88, 120, 184, 216, 0, 24], [88, 120, 184, 216, 8, 40], [88, 120, 184, 216, 24, 56], [88, 120, 184, 216, 40, 72], [88, 120, 184, 216, 56, 88], [88, 120, 184, 216, 72, 104], [88, 120, 184, 216, 88, 120], [88, 120, 184, 216, 104, 128], [88, 120, 200, 232, 0, 24], [88, 120, 200, 232, 8, 40], [88, 120, 200, 232, 24, 56], [88, 120, 200, 232, 40, 72], [88, 120, 200, 232, 56, 88], [88, 120, 200, 232, 72, 104], [88, 120, 200, 232, 88, 120], [88, 120, 200, 232, 104, 128], [88, 120, 216, 248, 0, 24], [88, 120, 216, 248, 8, 40], [88, 120, 216, 248, 24, 56], [88, 120, 216, 248, 40, 72], [88, 120, 216, 248, 56, 88], [88, 120, 216, 248, 72, 104], [88, 120, 216, 248, 88, 120], [88, 120, 216, 248, 104, 128], [88, 120, 216, 256, 88, 128], [88, 120, 232, 256, 0, 24], [88, 120, 232, 256, 8, 40], [88, 120, 232, 256, 24, 56], [88, 120, 232, 256, 40, 72], [88, 120, 232, 256, 56, 88], [88, 120, 232, 256, 72, 104], [88, 120, 232, 256, 88, 120], [88, 120, 232, 256, 104, 128], [88, 125, 0, 40, 0, 40], [88, 125, 0, 40, 24, 72], [88, 125, 0, 40, 56, 104], [88, 125, 0, 40, 88, 128], [88, 125, 8, 40, 88, 128], [88, 125, 24, 72, 0, 40], [88, 125, 24, 72, 24, 72], [88, 125, 24, 72, 40, 72], [88, 125, 24, 72, 56, 104], [88, 125, 24, 72, 88, 128], [88, 125, 24, 72, 104, 128], [88, 125, 56, 104, 0, 24], [88, 125, 56, 104, 0, 40], [88, 125, 56, 104, 24, 72], [88, 125, 56, 104, 56, 104], [88, 125, 56, 104, 88, 128], [88, 125, 56, 104, 104, 128], [88, 125, 72, 104, 40, 72], [88, 125, 72, 104, 56, 88], [88, 125, 88, 136, 0, 40], [88, 125, 88, 136, 24, 72], [88, 125, 88, 136, 56, 104], [88, 125, 88, 136, 88, 128], [88, 125, 120, 168, 0, 40], [88, 125, 120, 168, 8, 40], [88, 125, 120, 168, 24, 72], [88, 125, 120, 168, 56, 104], [88, 125, 120, 168, 88, 128], [88, 125, 136, 168, 8, 40], [88, 125, 136, 168, 24, 72], [88, 125, 152, 200, 0, 40], [88, 125, 152, 200, 24, 72], [88, 125, 152, 200, 56, 104], [88, 125, 152, 200, 88, 128], [88, 125, 152, 200, 104, 128], [88, 125, 168, 200, 24, 56], [88, 125, 168, 200, 24, 72], [88, 125, 184, 232, 0, 40], [88, 125, 184, 232, 24, 72], [88, 125, 184, 232, 56, 104], [88, 125, 184, 232, 88, 128], [88, 125, 200, 232, 40, 72], [88, 125, 200, 232, 56, 104], [88, 125, 216, 256, 0, 40], [88, 125, 216, 256, 24, 72], [88, 125, 216, 256, 56, 104], [88, 125, 216, 256, 88, 128], [92, 116, 0, 20, 0, 20], [92, 116, 0, 20, 0, 36], [92, 116, 0, 20, 12, 36], [92, 116, 0, 20, 28, 52], [92, 116, 0, 20, 44, 68], [92, 116, 0, 20, 60, 84], [92, 116, 0, 20, 76, 100], [92, 116, 0, 20, 92, 116], [92, 116, 0, 20, 108, 128], [92, 116, 0, 36, 0, 36], [92, 116, 12, 36, 0, 20], [92, 116, 12, 36, 12, 36], [92, 116, 12, 36, 28, 52], [92, 116, 12, 36, 44, 68], [92, 116, 12, 36, 60, 84], [92, 116, 12, 36, 76, 100], [92, 116, 12, 36, 92, 116], [92, 116, 12, 36, 108, 128], [92, 116, 28, 52, 0, 20], [92, 116, 28, 52, 12, 36], [92, 116, 28, 52, 28, 52], [92, 116, 28, 52, 44, 68], [92, 116, 28, 52, 60, 84], [92, 116, 28, 52, 76, 100], [92, 116, 28, 52, 92, 116], [92, 116, 28, 52, 108, 128], [92, 116, 28, 68, 60, 84], [92, 116, 28, 68, 60, 100], [92, 116, 44, 68, 0, 20], [92, 116, 44, 68, 12, 36], [92, 116, 44, 68, 28, 52], [92, 116, 44, 68, 44, 68], [92, 116, 44, 68, 60, 84], [92, 116, 44, 68, 76, 100], [92, 116, 44, 68, 92, 116], [92, 116, 44, 68, 108, 128], [92, 116, 60, 84, 0, 20], [92, 116, 60, 84, 12, 36], [92, 116, 60, 84, 28, 52], [92, 116, 60, 84, 44, 68], [92, 116, 60, 84, 60, 84], [92, 116, 60, 84, 76, 100], [92, 116, 60, 84, 92, 116], [92, 116, 60, 84, 108, 128], [92, 116, 76, 100, 0, 20], [92, 116, 76, 100, 12, 36], [92, 116, 76, 100, 28, 52], [92, 116, 76, 100, 44, 68], [92, 116, 76, 100, 60, 84], [92, 116, 76, 100, 76, 100], [92, 116, 76, 100, 92, 116], [92, 116, 76, 100, 108, 128], [92, 116, 92, 116, 0, 20], [92, 116, 92, 116, 12, 36], [92, 116, 92, 116, 28, 52], [92, 116, 92, 116, 44, 68], [92, 116, 92, 116, 60, 84], [92, 116, 92, 116, 76, 100], [92, 116, 92, 116, 92, 116], [92, 116, 92, 116, 108, 128], [92, 116, 92, 132, 0, 36], [92, 116, 108, 132, 0, 20], [92, 116, 108, 132, 0, 36], [92, 116, 108, 132, 12, 36], [92, 116, 108, 132, 28, 52], [92, 116, 108, 132, 44, 68], [92, 116, 108, 132, 60, 84], [92, 116, 108, 132, 76, 100], [92, 116, 108, 132, 92, 116], [92, 116, 108, 132, 108, 128], [92, 116, 124, 148, 0, 20], [92, 116, 124, 148, 0, 36], [92, 116, 124, 148, 12, 36], [92, 116, 124, 148, 28, 52], [92, 116, 124, 148, 44, 68], [92, 116, 124, 148, 60, 84], [92, 116, 124, 148, 76, 100], [92, 116, 124, 148, 92, 116], [92, 116, 124, 148, 108, 128], [92, 116, 140, 164, 0, 20], [92, 116, 140, 164, 12, 36], [92, 116, 140, 164, 28, 52], [92, 116, 140, 164, 44, 68], [92, 116, 140, 164, 60, 84], [92, 116, 140, 164, 76, 100], [92, 116, 140, 164, 92, 116], [92, 116, 140, 164, 108, 128], [92, 116, 156, 180, 0, 20], [92, 116, 156, 180, 12, 36], [92, 116, 156, 180, 28, 52], [92, 116, 156, 180, 44, 68], [92, 116, 156, 180, 60, 84], [92, 116, 156, 180, 76, 100], [92, 116, 156, 180, 92, 116], [92, 116, 156, 180, 108, 128], [92, 116, 172, 196, 0, 20], [92, 116, 172, 196, 12, 36], [92, 116, 172, 196, 28, 52], [92, 116, 172, 196, 44, 68], [92, 116, 172, 196, 60, 84], [92, 116, 172, 196, 76, 100], [92, 116, 172, 196, 92, 116], [92, 116, 172, 196, 108, 128], [92, 116, 188, 212, 0, 20], [92, 116, 188, 212, 12, 36], [92, 116, 188, 212, 28, 52], [92, 116, 188, 212, 44, 68], [92, 116, 188, 212, 60, 84], [92, 116, 188, 212, 76, 100], [92, 116, 188, 212, 92, 116], [92, 116, 188, 212, 108, 128], [92, 116, 188, 228, 0, 36], [92, 116, 188, 228, 92, 116], [92, 116, 188, 228, 92, 128], [92, 116, 204, 228, 0, 20], [92, 116, 204, 228, 12, 36], [92, 116, 204, 228, 28, 52], [92, 116, 204, 228, 44, 68], [92, 116, 204, 228, 60, 84], [92, 116, 204, 228, 76, 100], [92, 116, 204, 228, 92, 116], [92, 116, 204, 228, 108, 128], [92, 116, 220, 244, 0, 20], [92, 116, 220, 244, 12, 36], [92, 116, 220, 244, 28, 52], [92, 116, 220, 244, 44, 68], [92, 116, 220, 244, 60, 84], [92, 116, 220, 244, 76, 100], [92, 116, 220, 244, 92, 116], [92, 116, 220, 244, 108, 128], [92, 116, 236, 256, 0, 20], [92, 116, 236, 256, 12, 36], [92, 116, 236, 256, 28, 52], [92, 116, 236, 256, 44, 68], [92, 116, 236, 256, 60, 84], [92, 116, 236, 256, 76, 100], [92, 116, 236, 256, 92, 116], [92, 116, 236, 256, 108, 128], [92, 125, 0, 36, 0, 36], [92, 125, 0, 36, 28, 68], [92, 125, 0, 36, 60, 100], [92, 125, 0, 36, 92, 116], [92, 125, 0, 36, 92, 128], [92, 125, 12, 36, 28, 52], [92, 125, 28, 52, 0, 36], [92, 125, 28, 68, 0, 36], [92, 125, 28, 68, 28, 68], [92, 125, 28, 68, 60, 100], [92, 125, 28, 68, 92, 128], [92, 125, 60, 84, 60, 84], [92, 125, 60, 100, 0, 36], [92, 125, 60, 100, 12, 36], [92, 125, 60, 100, 28, 68], [92, 125, 60, 100, 60, 100], [92, 125, 60, 100, 92, 128], [92, 125, 76, 100, 12, 36], [92, 125, 92, 116, 108, 128], [92, 125, 92, 132, 0, 36], [92, 125, 92, 132, 28, 68], [92, 125, 92, 132, 60, 100], [92, 125, 92, 132, 76, 100], [92, 125, 92, 132, 92, 128], [92, 125, 124, 148, 44, 68], [92, 125, 124, 148, 60, 84], [92, 125, 124, 164, 0, 36], [92, 125, 124, 164, 28, 68], [92, 125, 124, 164, 44, 68], [92, 125, 124, 164, 60, 100], [92, 125, 124, 164, 92, 128], [92, 125, 124, 164, 108, 128], [92, 125, 124, 196, 0, 68], [92, 125, 156, 180, 28, 52], [92, 125, 156, 196, 0, 36], [92, 125, 156, 196, 28, 52], [92, 125, 156, 196, 28, 68], [92, 125, 156, 196, 60, 84], [92, 125, 156, 196, 60, 100], [92, 125, 156, 196, 92, 128], [92, 125, 188, 228, 0, 36], [92, 125, 188, 228, 28, 68], [92, 125, 188, 228, 60, 100], [92, 125, 188, 228, 92, 128], [92, 125, 220, 244, 92, 128], [92, 125, 220, 256, 0, 36], [92, 125, 220, 256, 28, 52], [92, 125, 220, 256, 28, 68], [92, 125, 220, 256, 60, 100], [92, 125, 220, 256, 92, 128], [92, 125, 236, 256, 0, 36], [104, 125, 0, 24, 0, 24], [104, 125, 0, 24, 8, 40], [104, 125, 0, 24, 24, 56], [104, 125, 0, 24, 40, 72], [104, 125, 0, 24, 56, 88], [104, 125, 0, 24, 72, 104], [104, 125, 0, 24, 88, 120], [104, 125, 0, 24, 104, 128], [104, 125, 8, 40, 0, 24], [104, 125, 8, 40, 8, 40], [104, 125, 8, 40, 24, 56], [104, 125, 8, 40, 40, 72], [104, 125, 8, 40, 56, 88], [104, 125, 8, 40, 72, 104], [104, 125, 8, 40, 88, 120], [104, 125, 8, 40, 104, 128], [104, 125, 24, 56, 0, 24], [104, 125, 24, 56, 8, 40], [104, 125, 24, 56, 24, 56], [104, 125, 24, 56, 40, 72], [104, 125, 24, 56, 56, 88], [104, 125, 24, 56, 72, 104], [104, 125, 24, 56, 88, 120], [104, 125, 24, 56, 104, 128], [104, 125, 40, 72, 0, 24], [104, 125, 40, 72, 8, 40], [104, 125, 40, 72, 24, 56], [104, 125, 40, 72, 40, 72], [104, 125, 40, 72, 56, 88], [104, 125, 40, 72, 72, 104], [104, 125, 40, 72, 88, 120], [104, 125, 40, 72, 104, 128], [104, 125, 56, 88, 0, 24], [104, 125, 56, 88, 8, 40], [104, 125, 56, 88, 24, 56], [104, 125, 56, 88, 40, 72], [104, 125, 56, 88, 56, 88], [104, 125, 56, 88, 72, 104], [104, 125, 56, 88, 88, 120], [104, 125, 56, 88, 104, 128], [104, 125, 72, 104, 0, 24], [104, 125, 72, 104, 8, 40], [104, 125, 72, 104, 24, 56], [104, 125, 72, 104, 40, 72], [104, 125, 72, 104, 56, 88], [104, 125, 72, 104, 72, 104], [104, 125, 72, 104, 88, 120], [104, 125, 72, 104, 104, 128], [104, 125, 88, 120, 0, 24], [104, 125, 88, 120, 8, 40], [104, 125, 88, 120, 24, 56], [104, 125, 88, 120, 40, 72], [104, 125, 88, 120, 56, 88], [104, 125, 88, 120, 72, 104], [104, 125, 88, 120, 88, 120], [104, 125, 88, 120, 104, 128], [104, 125, 104, 136, 0, 24], [104, 125, 104, 136, 8, 40], [104, 125, 104, 136, 24, 56], [104, 125, 104, 136, 40, 72], [104, 125, 104, 136, 56, 88], [104, 125, 104, 136, 72, 104], [104, 125, 104, 136, 88, 120], [104, 125, 104, 136, 104, 128], [104, 125, 120, 152, 0, 24], [104, 125, 120, 152, 8, 40], [104, 125, 120, 152, 24, 56], [104, 125, 120, 152, 40, 72], [104, 125, 120, 152, 56, 88], [104, 125, 120, 152, 72, 104], [104, 125, 120, 152, 88, 120], [104, 125, 120, 152, 104, 128], [104, 125, 136, 168, 0, 24], [104, 125, 136, 168, 8, 40], [104, 125, 136, 168, 24, 56], [104, 125, 136, 168, 40, 72], [104, 125, 136, 168, 56, 88], [104, 125, 136, 168, 72, 104], [104, 125, 136, 168, 88, 120], [104, 125, 136, 168, 104, 128], [104, 125, 152, 184, 0, 24], [104, 125, 152, 184, 8, 40], [104, 125, 152, 184, 24, 56], [104, 125, 152, 184, 40, 72], [104, 125, 152, 184, 56, 88], [104, 125, 152, 184, 72, 104], [104, 125, 152, 184, 88, 120], [104, 125, 152, 184, 104, 128], [104, 125, 168, 200, 0, 24], [104, 125, 168, 200, 8, 40], [104, 125, 168, 200, 24, 56], [104, 125, 168, 200, 40, 72], [104, 125, 168, 200, 56, 88], [104, 125, 168, 200, 72, 104], [104, 125, 168, 200, 88, 120], [104, 125, 168, 200, 104, 128], [104, 125, 184, 216, 0, 24], [104, 125, 184, 216, 8, 40], [104, 125, 184, 216, 24, 56], [104, 125, 184, 216, 40, 72], [104, 125, 184, 216, 56, 88], [104, 125, 184, 216, 72, 104], [104, 125, 184, 216, 88, 120], [104, 125, 184, 216, 104, 128], [104, 125, 200, 232, 0, 24], [104, 125, 200, 232, 8, 40], [104, 125, 200, 232, 24, 56], [104, 125, 200, 232, 40, 72], [104, 125, 200, 232, 56, 88], [104, 125, 200, 232, 72, 104], [104, 125, 200, 232, 88, 120], [104, 125, 200, 232, 104, 128], [104, 125, 216, 248, 0, 24], [104, 125, 216, 248, 8, 40], [104, 125, 216, 248, 24, 56], [104, 125, 216, 248, 40, 72], [104, 125, 216, 248, 56, 88], [104, 125, 216, 248, 72, 104], [104, 125, 216, 248, 88, 120], [104, 125, 216, 248, 104, 128], [104, 125, 216, 256, 0, 40], [104, 125, 232, 256, 0, 24], [104, 125, 232, 256, 8, 40], [104, 125, 232, 256, 24, 56], [104, 125, 232, 256, 40, 72], [104, 125, 232, 256, 56, 88], [104, 125, 232, 256, 72, 104], [104, 125, 232, 256, 88, 120], [104, 125, 232, 256, 104, 128], [108, 125, 0, 20, 0, 20], [108, 125, 0, 20, 12, 36], [108, 125, 0, 20, 28, 52], [108, 125, 0, 20, 44, 68], [108, 125, 0, 20, 60, 84], [108, 125, 0, 20, 76, 100], [108, 125, 0, 20, 92, 116], [108, 125, 0, 20, 108, 128], [108, 125, 12, 36, 0, 20], [108, 125, 12, 36, 12, 36], [108, 125, 12, 36, 28, 52], [108, 125, 12, 36, 44, 68], [108, 125, 12, 36, 60, 84], [108, 125, 12, 36, 76, 100], [108, 125, 12, 36, 92, 116], [108, 125, 12, 36, 108, 128], [108, 125, 28, 52, 0, 20], [108, 125, 28, 52, 12, 36], [108, 125, 28, 52, 28, 52], [108, 125, 28, 52, 44, 68], [108, 125, 28, 52, 60, 84], [108, 125, 28, 52, 76, 100], [108, 125, 28, 52, 92, 116], [108, 125, 28, 52, 108, 128], [108, 125, 44, 68, 0, 20], [108, 125, 44, 68, 12, 36], [108, 125, 44, 68, 28, 52], [108, 125, 44, 68, 44, 68], [108, 125, 44, 68, 60, 84], [108, 125, 44, 68, 76, 100], [108, 125, 44, 68, 92, 116], [108, 125, 44, 68, 108, 128], [108, 125, 60, 84, 0, 20], [108, 125, 60, 84, 12, 36], [108, 125, 60, 84, 28, 52], [108, 125, 60, 84, 44, 68], [108, 125, 60, 84, 60, 84], [108, 125, 60, 84, 76, 100], [108, 125, 60, 84, 92, 116], [108, 125, 60, 84, 108, 128], [108, 125, 60, 100, 108, 128], [108, 125, 76, 100, 0, 20], [108, 125, 76, 100, 12, 36], [108, 125, 76, 100, 28, 52], [108, 125, 76, 100, 44, 68], [108, 125, 76, 100, 60, 84], [108, 125, 76, 100, 76, 100], [108, 125, 76, 100, 92, 116], [108, 125, 76, 100, 108, 128], [108, 125, 92, 116, 0, 20], [108, 125, 92, 116, 12, 36], [108, 125, 92, 116, 28, 52], [108, 125, 92, 116, 44, 68], [108, 125, 92, 116, 60, 84], [108, 125, 92, 116, 76, 100], [108, 125, 92, 116, 92, 116], [108, 125, 92, 116, 108, 128], [108, 125, 108, 132, 0, 20], [108, 125, 108, 132, 12, 36], [108, 125, 108, 132, 28, 52], [108, 125, 108, 132, 44, 68], [108, 125, 108, 132, 60, 84], [108, 125, 108, 132, 76, 100], [108, 125, 108, 132, 92, 116], [108, 125, 108, 132, 108, 128], [108, 125, 124, 148, 0, 20], [108, 125, 124, 148, 12, 36], [108, 125, 124, 148, 28, 52], [108, 125, 124, 148, 44, 68], [108, 125, 124, 148, 60, 84], [108, 125, 124, 148, 76, 100], [108, 125, 124, 148, 92, 116], [108, 125, 124, 148, 108, 128], [108, 125, 140, 164, 0, 20], [108, 125, 140, 164, 12, 36], [108, 125, 140, 164, 28, 52], [108, 125, 140, 164, 44, 68], [108, 125, 140, 164, 60, 84], [108, 125, 140, 164, 76, 100], [108, 125, 140, 164, 92, 116], [108, 125, 140, 164, 108, 128], [108, 125, 156, 180, 0, 20], [108, 125, 156, 180, 12, 36], [108, 125, 156, 180, 28, 52], [108, 125, 156, 180, 44, 68], [108, 125, 156, 180, 60, 84], [108, 125, 156, 180, 76, 100], [108, 125, 156, 180, 92, 116], [108, 125, 156, 180, 108, 128], [108, 125, 172, 196, 0, 20], [108, 125, 172, 196, 12, 36], [108, 125, 172, 196, 28, 52], [108, 125, 172, 196, 44, 68], [108, 125, 172, 196, 60, 84], [108, 125, 172, 196, 76, 100], [108, 125, 172, 196, 92, 116], [108, 125, 172, 196, 108, 128], [108, 125, 188, 212, 0, 20], [108, 125, 188, 212, 12, 36], [108, 125, 188, 212, 28, 52], [108, 125, 188, 212, 44, 68], [108, 125, 188, 212, 60, 84], [108, 125, 188, 212, 76, 100], [108, 125, 188, 212, 92, 116], [108, 125, 188, 212, 108, 128], [108, 125, 204, 228, 0, 20], [108, 125, 204, 228, 12, 36], [108, 125, 204, 228, 28, 52], [108, 125, 204, 228, 44, 68], [108, 125, 204, 228, 60, 84], [108, 125, 204, 228, 76, 100], [108, 125, 204, 228, 92, 116], [108, 125, 204, 228, 108, 128], [108, 125, 220, 244, 0, 20], [108, 125, 220, 244, 12, 36], [108, 125, 220, 244, 28, 52], [108, 125, 220, 244, 44, 68], [108, 125, 220, 244, 60, 84], [108, 125, 220, 244, 76, 100], [108, 125, 220, 244, 92, 116], [108, 125, 220, 244, 108, 128], [108, 125, 236, 256, 0, 20], [108, 125, 236, 256, 12, 36], [108, 125, 236, 256, 28, 52], [108, 125, 236, 256, 44, 68], [108, 125, 236, 256, 60, 84], [108, 125, 236, 256, 76, 100], [108, 125, 236, 256, 92, 116], [108, 125, 236, 256, 108, 128]]

Режим: quick форма: (24, 32, 24) повторы: 2


## Измерения

In [3]:
result={'schema_version':1,'mode':MODE,'stage':STAGE,'shape':list(shape),
        'repetitions':repeats,'input_sha256':input_hash,
        'created_utc':datetime.now(timezone.utc).isoformat(),
        'environment':machine_environment(),
        'parallel_chunk':PARALLEL_CHUNK,
        'calibration':{},'test':{},'parallel':{},'discrepancy':None}
if MODE=='quick' or STAGE=='calibration':
    p=next(iter(inputs.values()))
    if MODE=='full':boxes=[tuple(b) for b in REFERENCE_BOXES]
    else:boxes=list(dict.fromkeys([full_box]+[r['box'] for a,m in geometries for r in windows(shape,a,m)]))
    result['calibration']=calibration_passes(p,target,boxes,repeats)
    print('Записей по окнам:',len(result['calibration']))
if MODE=='quick' or STAGE=='test':
    for ck,p in inputs.items():
        a,m=geometries[0] if MODE=='quick' else (32,4)
        variants={'full':full,'all_windows':windows(shape,a,m)}
        result['test'][ck]=direct_passes(p,target,variants,repeats)
        if ck in ['4000','synthetic']:
            a2,m2=geometries[-1] if MODE=='quick' else (64,4)
            result['test'][ck]['larger_windows']=direct_passes(p,target,{'all_windows':windows(shape,a2,m2)},repeats)['all_windows']
            if RUN_PARALLEL:
                result['parallel']={name:parallel_passes(p,target,regs,repeats,PARALLEL_CHUNK)
                                    for name,regs in variants.items()}
        print(ck,{name:round(values['wall_median_s'],6) for name,values in result['test'][ck].items()})

Записей по окнам: 45
synthetic {'full': 0.088944, 'all_windows': 0.125443, 'larger_windows': 0.084016}


## Неэквивалентность полной и оконной функций

In [4]:
def loss_gradient(p,y,regs):
    """Собрать функцию и градиент по f=1-p, с принадлежностью по рождению в собственной части."""
    f=1.0-p.astype(np.float64);g=1.0-y.astype(np.float64)
    grad=np.zeros(p.size,dtype=np.float64);loss=0.0
    def arr(value):return np.asarray(value,dtype=np.int64).reshape(-1,3)
    def inside(coords,core):
        lo=np.array(core[0::2]);hi=np.array(core[1::2])
        return ((coords>=lo)&(coords<hi)).all(axis=1)
    def add(coords,values):
        np.add.at(grad,np.ravel_multi_index(tuple(coords.T),p.shape),values)
    for reg in regs:
        r=call_matching(p,y,reg['box']);off=np.array(reg['box'][0::2])
        for dim in range(len(r.num_matched)):
            bp=arr(r.input1_matched_birth_coordinates[dim])+off
            dp=arr(r.input1_matched_death_coordinates[dim])+off
            by=arr(r.input2_matched_birth_coordinates[dim])+off
            dy=arr(r.input2_matched_death_coordinates[dim])+off
            keep=inside(bp,reg['core']);bp,dp,by,dy=bp[keep],dp[keep],by[keep],dy[keep]
            db=f[tuple(bp.T)]-g[tuple(by.T)];dd=f[tuple(dp.T)]-g[tuple(dy.T)]
            loss+=float(2*np.sum(db**2+dd**2));add(bp,4*db);add(dp,4*dd)
            ub=arr(r.input1_unmatched_birth_coordinates[dim])+off
            ud=arr(r.input1_unmatched_death_coordinates[dim])+off
            keep=inside(ub,reg['core']);ub,ud=ub[keep],ud[keep]
            delta=f[tuple(ub.T)]-f[tuple(ud.T)]
            loss+=float(np.sum(delta**2));add(ub,2*delta);add(ud,-2*delta)
    return loss,grad

if RUN_DISCREPANCY or MODE=='quick':
    key='synthetic' if MODE=='quick' else '4000'
    p=inputs[key];a,m=geometries[0] if MODE=='quick' else (32,4)
    lf,gf=loss_gradient(p,target,full)
    lw,gw=loss_gradient(p,target,windows(shape,a,m))
    nf,nw=np.linalg.norm(gf),np.linalg.norm(gw)
    metrics={'L_ratio':lw/lf if lf else None,
             'cos':float(np.dot(gf,gw)/(nf*nw)) if nf*nw else None,
             'rel_l2':float(np.linalg.norm(gw-gf)/nf) if nf else None}
    result['discrepancy']={'checkpoint':key,'core_side':a,'margin':m,**metrics}
    print('Контроль функции и градиента (вне таймера):',metrics)
    if MODE=='quick':
        # Одно окно на весь массив обязано совпасть с полным расчётом.
        lcheck,gcheck=loss_gradient(p,target,windows(shape,max(shape),0))
        assert np.isclose(lcheck,lf) and np.array_equal(gcheck,gf)
else:print('Контроль функции и градиента отключён.')

Контроль функции и градиента (вне таймера): {'L_ratio': 0.9733090763451038, 'cos': 0.9392332183424115, 'rel_l2': 0.34667051685976513}


In [5]:
if SAVE_RESULTS:
    OUTPUT.mkdir(parents=True,exist_ok=True)
    name='measurement_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')+'.json'
    (OUTPUT/name).write_text(json.dumps(result,ensure_ascii=False,indent=2,allow_nan=False))
    print('Результат сохранён в отдельный файл нового прогона.')
else:print('SAVE_RESULTS=False: сохранённые результаты статьи не изменены.')

SAVE_RESULTS=False: сохранённые результаты статьи не изменены.
